# AI574 — Multi-Domain Intelligent Assistant with Supervisor Agent Architecture

### Team 3
### Rosh Sreedharan and Dhruv Chaudhry

### School of Graduate Professional Studies

### Data Analytics
### AI 574 – Natural Language Processing 
### Spring 2026


Submission notebook for a multi-domain Corrective RAG assistant across industrial, recipe, and scientific corpora.

Reported 150-query result: compared with monolithic RAG, CRAG is slower but reduces unblocked hallucinations from **19.3%** to **4.7%** while keeping **100.0%** route accuracy and **100.0%** retrieval P@5.


<a id="project-framing"></a>

## Project Framing

This project routes each query to the right domain before retrieval and generation. The three domains are:

- **Industrial:** PLC/SCADA, drives, fault codes, and maintenance safety.
- **Recipe:** substitutions, cooking guidance, recipes, and nutrition tags.
- **Scientific:** ArXiv-backed paper search and summarization.

The main system is `multi_agent_crag`: routing, domain retrieval, document grading, optional rewrite, grounded generation, and hallucination validation.

The quantitative baseline is `monolithic_rag`: one pooled retriever across all domains, then generate.


<a id="system-architecture-at-a-glance"></a>

## System Architecture

Single-query flow:

`query -> router/supervisor -> domain retriever -> CRAG checks -> grounded answer`

Extra paths: synthesis combines multiple domains; fallback uses web search for out-of-scope questions.

Evaluation compares `multi_agent_crag` against `monolithic_rag` on 150 labeled queries.


<a id="table-of-contents-submission"></a>

## Table of Contents

- **Setup:** §1–§8
- **Indexing:** §8.5–§11
- **Workflow and demo:** §12–§12.5
- **Baselines and evaluation:** §20–§22
- **Validation and summary:** §23–§25


<a id="how-to-use-this-submission-notebook"></a>

## How to Use

1. Run setup through §8.
2. Restore the Chroma snapshot; rebuild indexes only if needed.
3. Build the workflow and run the five-query demo in §12.5.
4. Use the cached 150-query results in §21.7–§22 unless rerunning the benchmark intentionally.


<a id="1-clone-update-repo"></a>

## 1. Clone / update repo


In [1]:
# Purpose: clone or update the project repository inside the Colab runtime.
import os

# If the AI574 directory does not exist in /content, clone the repository from GitHub
if not os.path.exists('/content/AI574'):
    !git clone https://github.com/roshcs/AI574.git

# Change the current working directory to /content/AI574
%cd /content/AI574

# Pull the latest changes from the repository
!git pull


/content/AI574
Already up to date.


<a id="2-gpu-google-drive-chroma-persistence"></a>

## 2. GPU + Google Drive + Chroma persistence


### Setup & Drive persistence

The next cell mounts Google Drive, sets up `PROJECT_DIR`, and points `CHROMA_PERSIST_DIR` at `PROJECT_DIR/chroma_db` on Drive. This means the vector index is written to Drive and **reused across sessions** — you only need to run full recipe indexing once.

If you change the persist path, do **Runtime → Restart session** then **Run all** so `CONFIG` picks up the new value.


In [2]:
# Purpose: verify GPU access, mount Drive, and configure persistent project paths.
import torch

# Check CUDA (GPU) availability and print details about the GPU and its memory.
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

from google.colab import drive
# Mount Google Drive to allow persistence across sessions.
drive.mount('/content/drive')

# Create a persistent project directory in Google Drive for all AI574 artifacts.
import os
PROJECT_DIR = '/content/drive/MyDrive/AI574_Multi_Domain_Agent'
os.makedirs(PROJECT_DIR, exist_ok=True)                # Ensure main project directory exists
os.makedirs(f'{PROJECT_DIR}/data', exist_ok=True)      # Create subdirectory for data
os.makedirs(f'{PROJECT_DIR}/models', exist_ok=True)    # Create subdirectory for models
print(f'Project directory: {PROJECT_DIR}')

# Set up ChromaDB directories:
#   - Live Chroma index lives on the Colab VM's local SSD (faster, safer for SQLite)
#   - Backups/snapshots live in Google Drive only (tar.gz created by helpers in §2.5)
CHROMA_DIR = "/content/chroma_db"
CHROMA_BACKUP_DIR = os.path.join(PROJECT_DIR, "chroma_backups")
os.makedirs(CHROMA_DIR, exist_ok=True)                 # Local SSD for live Chroma
os.makedirs(CHROMA_BACKUP_DIR, exist_ok=True)          # Drive for Chroma DB backups
os.environ["CHROMA_PERSIST_DIR"] = CHROMA_DIR          # Export live Chroma location for other tools
print(f'ChromaDB live dir:    {CHROMA_DIR} (local SSD)')
print(f'ChromaDB backup dir:  {CHROMA_BACKUP_DIR} (Drive, tar.gz snapshots)')


CUDA available: True
GPU: NVIDIA A100-SXM4-80GB
Memory: 85.1 GB
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project directory: /content/drive/MyDrive/AI574_Multi_Domain_Agent
ChromaDB live dir:    /content/chroma_db (local SSD)
ChromaDB backup dir:  /content/drive/MyDrive/AI574_Multi_Domain_Agent/chroma_backups (Drive, tar.gz snapshots)


<a id="25-chroma-safety-helpers"></a>

## 2.5 Chroma safety helpers

Defines `verify_chroma`, `snapshot_chroma`, `restore_chroma_from_snapshot`, `try_recover_sqlite`, and `bootstrap_chroma`. These keep Chroma corruption-safe:

- Live DB stays on local SSD (`/content/chroma_db`), where SQLite's fsync works.
- Drive holds versioned tar.gz snapshots under `CHROMA_BACKUP_DIR` (rolling 3).
- `bootstrap_chroma()` is called once before the vector store is opened and will restore the newest snapshot into local disk if local is empty, or auto-recover if SQLite is malformed.
- `snapshot_chroma(label)` is called after each indexing cell so you always have a known-good archive on Drive.


In [3]:
# Purpose: define Chroma snapshot, restore, integrity-check, and recovery helpers.
import os
import shutil
import sqlite3
import subprocess
import tarfile
import datetime
import gc
from pathlib import Path

# Ensure the sqlite3 CLI tool is available in the Colab environment (needed for .recover operations)
def ensure_sqlite_cli() -> None:
    """Install the sqlite3 CLI once per session (needed for .recover)."""
    if shutil.which("sqlite3"):
        # Already installed; nothing to do
        return
    # Install sqlite3 using apt-get quietly
    subprocess.run(
        ["apt-get", "-y", "-qq", "install", "sqlite3"],
        check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    )

# Get the path to the main Chroma SQLite file
def _chroma_sqlite_path() -> Path:
    return Path(CHROMA_DIR) / "chroma.sqlite3"

# Verify the integrity of the Chroma SQLite database
def verify_chroma() -> tuple[bool, str]:
    """Return (ok, reason). Safe to call even if DB is absent."""
    # Chroma stores collection metadata in SQLite.
    # Running SQLite's integrity check catches any corruption before DB is used.
    db = _chroma_sqlite_path()
    if not db.exists():
        return False, "missing"
    try:
        con = sqlite3.connect(str(db))
        try:
            row = con.execute("PRAGMA integrity_check;").fetchone()
        finally:
            con.close()
        reason = row[0] if row else "no result"
        return (reason == "ok"), reason
    except sqlite3.DatabaseError as e:
        # Could not open or check DB - considered unhealthy
        return False, str(e)

# List all backup snapshot tarballs present in the backup directory
def _list_snapshots() -> list[Path]:
    return sorted(Path(CHROMA_BACKUP_DIR).glob("chroma_*.tar.gz"))

# Take a safe backup snapshot of the current Chroma DB; keep only `keep` latest
def snapshot_chroma(label: str = "", keep: int = 3) -> Path:
    """
    Checkpoint the WAL, verify integrity, tar the live Chroma dir, and atomically
    publish to Drive. Refuses to archive an unhealthy DB. Prunes to `keep` newest.
    """
    # Fold WAL pages into chroma.sqlite3 so the snapshot is up-to-date
    db = _chroma_sqlite_path()
    if db.exists():
        con = sqlite3.connect(str(db))
        try:
            con.execute("PRAGMA wal_checkpoint(TRUNCATE);")
        finally:
            con.close()

    # Only back up healthy DBs
    ok, reason = verify_chroma()
    if not ok:
        raise RuntimeError(f"Refusing to snapshot unhealthy Chroma DB: {reason}")

    # Write snapshot to a temporary file first; rename atomically for safety
    Path(CHROMA_BACKUP_DIR).mkdir(parents=True, exist_ok=True)
    stamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
    suffix = f"_{label}" if label else ""
    out = Path(CHROMA_BACKUP_DIR) / f"chroma_{stamp}{suffix}.tar.gz"
    tmp = out.with_suffix(".tar.gz.tmp")
    if tmp.exists():
        tmp.unlink()
    with tarfile.open(tmp, "w:gz") as tf:
        tf.add(CHROMA_DIR, arcname="chroma_db")
    tmp.replace(out)

    # Prune old snapshots: keep only `keep` latest
    snaps = _list_snapshots()
    for old in snaps[:-keep]:
        try:
            old.unlink()
            print(f"  pruned old snapshot: {old.name}")
        except Exception as e:
            print(f"  prune skipped for {old.name}: {e}")

    # Report where the snapshot was written
    size_mb = out.stat().st_size / 1e6
    print(f"Snapshot written: {out.name} ({size_mb:.1f} MB)")
    return out

# Restore a snapshot tarball as the current Chroma database
def restore_chroma_from_snapshot(snap: Path) -> None:
    """
    Replace /content/chroma_db with the contents of `snap` (a tar.gz written by
    snapshot_chroma). Any existing local dir is moved aside for post-mortem.
    """
    live = Path(CHROMA_DIR)
    # If the DB already exists, move it aside for debugging
    if live.exists() and any(live.iterdir()):
        stamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
        aside = Path(f"{CHROMA_DIR}.replaced-{stamp}")
        live.rename(aside)
        print(f"  existing local dir moved aside: {aside}")
    live.mkdir(parents=True, exist_ok=True)

    # Extract the snapshot tarball
    with tarfile.open(snap, "r:gz") as tf:
        tf.extractall("/content")

    # Remove any transient SQLite sidecar files that may cause issues on restart
    for ext in ("-wal", "-shm", "-journal"):
        p = Path(str(_chroma_sqlite_path()) + ext)
        if p.exists():
            p.unlink()
    print(f"  restored snapshot: {snap.name}")

# Attempt to repair a corrupted Chroma SQLite DB using the sqlite3 .recover command
def try_recover_sqlite() -> bool:
    """
    Attempt SQLite .recover on a malformed live DB. Returns True on success.
    Leaves the original file as .corrupt-<stamp> on swap.
    """
    ensure_sqlite_cli()  # install sqlite3 CLI if needed
    db = _chroma_sqlite_path()
    if not db.exists():
        return False
    recovered = db.with_suffix(db.suffix + ".recovered")
    if recovered.exists():
        recovered.unlink()
    try:
        # Attempt to salvage data with .recover
        dump = subprocess.check_output(
            ["sqlite3", str(db), ".recover"], stderr=subprocess.STDOUT,
        )
        subprocess.run(["sqlite3", str(recovered)], input=dump, check=True)
    except subprocess.CalledProcessError as e:
        print(f"  .recover failed: {e.output.decode('utf-8', 'replace')[-400:]}")
        return False
    if not recovered.exists() or recovered.stat().st_size == 0:
        return False
    # Rename the corrupt DB for forensics, move in the recovered version
    stamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
    db.rename(db.with_suffix(db.suffix + f".corrupt-{stamp}"))
    recovered.rename(db)
    # Also remove any corrupt SQLite sidecars
    for ext in ("-wal", "-shm", "-journal"):
        p = Path(str(db) + ext)
        if p.exists():
            p.unlink()
    print("  .recover produced a healthy DB and swapped it in")
    return True

# Prepare Chroma for use: ensure local DB is ready, auto-recover if needed
def bootstrap_chroma() -> None:
    """
    Call once before opening VectorStoreService. Ensures /content/chroma_db
    contains a healthy DB: restores newest snapshot if local is empty, or
    auto-recovers via .recover / snapshot restore if the DB is malformed.
    """
    ensure_sqlite_cli()
    live = Path(CHROMA_DIR)
    live.mkdir(parents=True, exist_ok=True)

    snaps = _list_snapshots()
    if not any(live.iterdir()):
        # No local DB: restore newest available snapshot from Drive, if any
        if snaps:
            print(f"Local Chroma empty; restoring newest snapshot ({snaps[-1].name})")
            restore_chroma_from_snapshot(snaps[-1])
        else:
            print("Local Chroma empty and no snapshot on Drive; starting fresh.")
            print("Run the indexing cells to populate, then snapshot_chroma(...)." )
            return

    # Check if the DB is healthy; if not, try to recover
    ok, reason = verify_chroma()
    if not ok:
        print(f"Local Chroma unhealthy: {reason}")
        # Prefer local .recover for minimal data loss; fall back to Drive snapshot if needed
        if try_recover_sqlite():
            pass
        elif snaps:
            print(f"Restoring newest snapshot ({snaps[-1].name})")
            restore_chroma_from_snapshot(snaps[-1])
        else:
            raise RuntimeError(
                "Local Chroma is corrupt and no snapshot exists on Drive. "
                "Wipe /content/chroma_db and re-index."
            )
        # Confirm recovery worked; if not, abort
        ok, reason = verify_chroma()
        if not ok:
            raise RuntimeError(f"Chroma still unhealthy after recovery: {reason}")

    print(f"Chroma ready at {CHROMA_DIR}")

# Print a summary of available snapshots for manual inspection / reassurance
ensure_sqlite_cli()
print(f"Snapshots on Drive: {len(_list_snapshots())} present in {CHROMA_BACKUP_DIR}")
for _snap in _list_snapshots()[-3:]:
    print(f"  - {_snap.name} ({_snap.stat().st_size / 1e6:.1f} MB)")


Snapshots on Drive: 4 present in /content/drive/MyDrive/AI574_Multi_Domain_Agent/chroma_backups
  - chroma_20260423-183556_recipe.tar.gz (13721.3 MB)
  - chroma_20260423-190552_scientific.tar.gz (13721.3 MB)
  - chroma_20260423-200647_seed.tar.gz (13721.3 MB)


<a id="3-install-dependencies"></a>

## 3. Install dependencies


In [4]:
# Action: Run this cell to install all packages required for the notebook.
# This includes all major libraries for LLMs, data loading, vector databases, and GPU support.

# %%capture  # Uncomment to suppress output if desired

# Install core and supporting LangChain, Keras, ChromaDB, and ML libraries.
# Important: keep each backslash as the final character on its line. Shell line
# continuations break if comments or spaces appear after the backslash.
!pip install -q \
  "langchain>=0.3.0" \
  "langchain-core>=0.3.0" \
  "langgraph>=0.2.0" \
  "langchain-text-splitters>=0.2.0" \
  "keras>=3.0" \
  "keras-hub>=0.17.0" \
  "sentence-transformers>=3.0.0" \
  "transformers>=4.45,<5.0" \
  "chromadb>=0.5.0" \
  "PyPDF2>=3.0.0" \
  "arxiv>=2.1.0" \
  "pydantic>=2.0" \
  "kagglehub>=0.3.0" \
  "kaggle>=1.6.0" \
  "ddgs>=9.0.0"

# Package roles:
# - LangChain/LangGraph: LLM workflow orchestration.
# - Keras/KerasHub/JAX: optional local model backend.
# - SentenceTransformers/Transformers: embeddings and model utilities.
# - ChromaDB/PyPDF2/ArXiv/Kaggle: corpus ingestion and vector storage.
# - ddgs: DuckDuckGo web-search fallback provider.

# Upgrade Kaggle and KaggleHub (ensure latest versions after specific installs above)
!pip install -q -U kagglehub kaggle

# Install JAX with CUDA 12 support (for high-speed GPU computation with JAX backend)
!pip install -q "jax[cuda12]" -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html

# Google GenAI integration for LangChain 
!pip install -q langchain-google-genai



<a id="4-set-keras-backend-and-verify"></a>

## 4. Set Keras backend and verify


In [5]:
# Set the Keras backend to JAX by modifying the environment variable
import os
os.environ['KERAS_BACKEND'] = 'jax'

# Import keras and verify that the backend is set correctly
import keras
print(f'Keras backend: {keras.backend.backend()}')  # Should print 'jax'
print(f'Keras version: {keras.__version__}')        # Print the Keras version to confirm import


Keras backend: jax
Keras version: 3.13.2


In [6]:
# Import essential libraries from core technology stack.
import langchain          # Main LangChain library for LLM workflows
import langgraph          # Graph-based orchestration for LangChain
import keras_hub          # Access to community Keras models
import chromadb           # Vector database for storage and retrieval
from sentence_transformers import SentenceTransformer  # Embeddings from Sentence Transformers
import jax                # JAX for high-performance array computing

# Verify that all imports were successful
print('All imports successful!')

# Print available JAX devices (CPU/GPU/TPU)
print(f'JAX devices: {jax.devices()}')


All imports successful!
JAX devices: [CudaDevice(id=0)]


<a id="5-project-import-path"></a>

## 5. Project import path


In [7]:
# Purpose: force notebook imports to use the latest cloned AI574 project code.
import os
import sys
import importlib
import subprocess
import inspect

# Path to the root of the cloned git repo (AI574)
GIT_REPO = '/content/AI574'

# Ensure the repo exists
if not os.path.isdir(GIT_REPO):
    raise RuntimeError(
        f"{GIT_REPO} not found. Run cell 0 first - it clones / git-pulls the repo."
    )

# Define legacy project root
LEGACY_ROOT = os.path.join(PROJECT_DIR, 'project')

# Remove any old legacy/project repo paths from sys.path
sys.path[:] = [p for p in sys.path if p not in (LEGACY_ROOT, GIT_REPO)]

# Names of top-level packages that will be force reloaded to avoid old code
PROJECT_PACKAGES = (
    'config', 'foundation', 'ingestion',
    'rag_core', 'agents', 'orchestration', 'evaluation',
)

# Remove any loaded modules from sys.modules that are within the project packages or project paths
for name in list(sys.modules):
    mod = sys.modules.get(name)
    f = getattr(mod, '__file__', '') or ''
    top = name.split('.')[0]
    if top in PROJECT_PACKAGES or f.startswith(LEGACY_ROOT) or f.startswith(GIT_REPO):
        del sys.modules[name]

# Invalidate import caches so that fresh code gets loaded
importlib.invalidate_caches()

# Ensure GIT_REPO is on sys.path so imports use latest code
if GIT_REPO not in sys.path:
    sys.path.insert(0, GIT_REPO)

# Make sure required data directories exist in the project root
for d in ('data/industrial', 'data/recipes', 'data/recipe', 'data/scientific'):
    os.makedirs(os.path.join(PROJECT_DIR, d), exist_ok=True)

# Print a short git SHA for current code version
try:
    sha = subprocess.check_output(
        ['git', '-C', GIT_REPO, 'rev-parse', '--short', 'HEAD'],
        stderr=subprocess.DEVNULL,
    ).decode().strip()
except Exception:
    sha = 'unknown'
print(f"PROJECT_ROOT = {GIT_REPO} (commit {sha})")
print("on sys.path:", GIT_REPO in sys.path)

# Check for the presence of all major project code directories
for pkg in PROJECT_PACKAGES:
    pth = os.path.join(GIT_REPO, pkg)
    print(f"  {pkg:>14}: {'OK' if os.path.isdir(pth) else 'MISSING'}")

# Import an example agent to prove that the path fix worked
import agents.industrial_agent
print(f"Found industrial agent: {agents.industrial_agent.__file__}")

# Import the IndexBuilder and check that its index_recipes method signature has the 'dedupe' parameter
from ingestion.index_builder import IndexBuilder
_sig = inspect.signature(IndexBuilder.index_recipes)
if 'dedupe' not in _sig.parameters:
    raise RuntimeError(
        "Loaded an OLD ingestion/index_builder.py (missing `dedupe`). "
        f"Loaded from: {inspect.getfile(IndexBuilder)}. "
        "Re-run cell 0 to `git pull`, then Runtime -> Restart session and Run all."
    )
print(f"index_recipes loaded from: {inspect.getfile(IndexBuilder)}")
print(f"index_recipes signature: {_sig}")


PROJECT_ROOT = /content/AI574 (commit fcc10eb)
on sys.path: True
          config: OK
      foundation: OK
       ingestion: OK
        rag_core: OK
          agents: OK
   orchestration: OK
      evaluation: OK
Found industrial agent: /content/AI574/agents/industrial_agent.py
index_recipes loaded from: /content/AI574/ingestion/index_builder.py
index_recipes signature: (self, csv_path: 'str', max_rows: 'Optional[int]' = None, *, dedupe: 'bool' = True, drop_junk: 'bool' = True, interactions_csv: 'Optional[str]' = None, batch_size: 'int' = 1000, skip_existing: 'bool' = True) -> 'int'


<a id="6-load-credentials-colab-secrets-getpass"></a>

## 6. Load credentials (Colab Secrets / getpass)


In [8]:
# Purpose: load API credentials from Colab Secrets or prompt securely when missing.
import os
import json
import pathlib
import stat
import getpass


def _try_userdata(name: str) -> None:
    """Load one Colab secret if present (missing secret is OK)."""
    try:
        from google.colab import userdata
        val = userdata.get(name)
        if val:
            os.environ[name] = str(val).strip()
    except Exception:
        pass


# Load secrets from Colab if available
for _secret in ("GOOGLE_API_KEY", "KAGGLE_API_TOKEN", "KAGGLE_USERNAME", "KAGGLE_KEY", "GROQ_API_KEY"):
    _try_userdata(_secret)

# Fallback prompts
if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter GOOGLE_API_KEY: ")

if not os.environ.get("KAGGLE_API_TOKEN") and not (
    os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY")
):
    print("No Kaggle token found. Enter API token or legacy username+key.")
    use_new = input("Paste KAGGLE_API_TOKEN, or press Enter for legacy credentials: ").strip()
    if use_new:
        os.environ["KAGGLE_API_TOKEN"] = use_new
    else:
        if not os.environ.get("KAGGLE_USERNAME"):
            os.environ["KAGGLE_USERNAME"] = input("KAGGLE_USERNAME: ").strip()
        if not os.environ.get("KAGGLE_KEY"):
            os.environ["KAGGLE_KEY"] = getpass.getpass("KAGGLE_KEY: ").strip()


def _sync_kaggle_files() -> None:
    """Write ~/.kaggle files so kagglehub and CLI use same auth."""
    kd = pathlib.Path.home() / ".kaggle"
    kd.mkdir(parents=True, exist_ok=True)
    token = os.environ.get("KAGGLE_API_TOKEN", "").strip()
    user = os.environ.get("KAGGLE_USERNAME", "").strip()
    key = os.environ.get("KAGGLE_KEY", "").strip()
    if token:
        p = kd / "access_token"
        p.write_text(token)
        p.chmod(stat.S_IRUSR | stat.S_IWUSR)
        print("Kaggle auth: API token synced")
    elif user and key:
        p = kd / "kaggle.json"
        p.write_text(json.dumps({"username": user, "key": key}))
        p.chmod(stat.S_IRUSR | stat.S_IWUSR)
        print(f"Kaggle auth: legacy creds synced for user '{user}'")
    else:
        print("WARNING: No Kaggle credentials to sync.")


_sync_kaggle_files()
print("GOOGLE_API_KEY loaded:", bool(os.environ.get("GOOGLE_API_KEY")))
print("GROQ_API_KEY loaded:", bool(os.environ.get("GROQ_API_KEY")))
print("KAGGLE_API_TOKEN loaded:", bool(os.environ.get("KAGGLE_API_TOKEN")))


No Kaggle token found. Enter API token or legacy username+key.
Kaggle auth: API token synced
GOOGLE_API_KEY loaded: True
GROQ_API_KEY loaded: False
KAGGLE_API_TOKEN loaded: True


<a id="7-load-llm"></a>

## 7. Load LLM


In [9]:
# Purpose: initialize the primary hosted LLM and optional local Gemma fallback.
import os
from config.settings import CONFIG

_CONSENT_URL = "https://www.kaggle.com/models/keras/gemma3/keras/gemma3_instruct_12b"
_HELP = (
    "Still 403 after accepting the license? Usually: (1) API credentials belong to a "
    "different Kaggle account than the browser session that accepted Gemma \u2014 copy "
    "Generate New Token (Settings \u2192 API) into Colab Secret KAGGLE_API_TOKEN, "
    "or use Legacy username+key for that same account; (2) re-run the credentials "
    "cell so ~/.kaggle/ is rewritten."
)

USE_GEMMA = False  # Set to True to enable local Gemma (KerasHub) fallback

llm = None
_gemini_error = None

if os.environ.get("GOOGLE_API_KEY"):
    try:
        from foundation.model_registry import get_llm
        llm = get_llm("gemini_flash")
        print("LLM loaded (gemini_flash, hosted)")
    except Exception as e:
        _gemini_error = e
        print(f"Gemini unavailable ({e}).")

if llm is None and USE_GEMMA:
    print("USE_GEMMA is True — attempting local Gemma (KerasHub)...")
    try:
        from foundation.llm_wrapper import KerasHubChatModel
        llm = KerasHubChatModel(config=CONFIG.llm)
        print("LLM loaded (KerasHub local Gemma)")
    except Exception as e:
        err = str(e).lower()
        if "403" in err or "permission" in err or "consent" in err:
            print("Kaggle returned 403 (often account mismatch or wrong token type).")
            print("  Accept Gemma once at:")
            print(f"  {_CONSENT_URL}")
            print("  " + _HELP)
        else:
            print(f"KerasHub load failed: {e}\n")

if llm is None:
    parts = []
    if _gemini_error is not None:
        parts.append(f"Gemini failed: {_gemini_error}")
    elif not os.environ.get("GOOGLE_API_KEY"):
        parts.append("GOOGLE_API_KEY is not set.")
    if not USE_GEMMA:
        parts.append("Local Gemma fallback is disabled (set USE_GEMMA = True to enable).")
    raise RuntimeError("No LLM available. " + " ".join(parts))


LLM loaded (gemini_flash, hosted)


<a id="8-embeddings-vector-store"></a>

## 8. Embeddings + vector store


In [10]:
# Import the EmbeddingService for generating embeddings.
# The project wraps `thenlper/gte-large` behind this service so every domain
# uses the same embedding space and can share the same Chroma configuration.
from foundation.embedding_service import EmbeddingService

# Initialize the embedding service once. This can take a moment on a fresh
# runtime because the model may need to download from Hugging Face.
embedder = EmbeddingService()

# Smoke test: embed one industrial-style query and print the vector dimension.
# gte-large should produce 1024-dimensional vectors; a different number usually
# means the wrong embedding model or stale package version loaded.
test_vec = embedder.embed_query("PLC fault code")
print(f"✅ Embedding dimension: {len(test_vec)}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


✅ Embedding dimension: 1024


In [11]:
# Purpose: restore or repair Chroma before opening the vector store.
# Prepare the local Chroma directory before VectorStoreService opens it.
# In Colab this restores the latest Drive snapshot when /content/chroma_db is
# empty and repairs/restores it if SQLite integrity checks fail.
bootstrap_chroma()


Chroma ready at /content/chroma_db


In [12]:
# Purpose: open the shared Chroma vector store after bootstrap has prepared it.
from foundation.vector_store import VectorStoreService

# VectorStoreService owns the three domain collections and exposes a common
# search API used by CRAG agents, the baseline, and validation cells.
vs = VectorStoreService(embedding_service=embedder)
print("✅ Vector store initialized")
# Print document counts so the notebook output proves which snapshot/corpus was
# available for this run.
print(vs.get_all_stats())


✅ Vector store initialized
[{'domain': 'industrial', 'collection_name': 'industrial_knowledge', 'document_count': 6150}, {'domain': 'recipe', 'collection_name': 'recipe_knowledge', 'document_count': 231311}, {'domain': 'scientific', 'collection_name': 'scientific_knowledge', 'document_count': 686968}]


<a id="85-indexing-control"></a>

## 8.5 Indexing control

These flags gate every indexing cell below. They default to `False` so a **Run all** after `bootstrap_chroma()` does **not** touch the restored snapshot. Flip only the domain(s) you want to rebuild, then re-run those cells (or **Run all** again).

The cell immediately after the flags always constructs `IndexBuilder` / `ChunkingPipeline` so recipe and scientific indexing still work when industrial reindex is off.

Sanity-check cells and bootstrap are always active and unaffected.


In [13]:
# Rebuild controls. Keep these False for normal demo runs so restored snapshots
# are reused. Set a flag to True only when intentionally regenerating that
# domain's Chroma collection from raw data.
REINDEX_INDUSTRIAL = False
REINDEX_RECIPE = False
REINDEX_SCIENTIFIC = False

print(
    f"Reindex flags -> industrial={REINDEX_INDUSTRIAL}  "
    f"recipe={REINDEX_RECIPE}  scientific={REINDEX_SCIENTIFIC}"
)


Reindex flags -> industrial=False  recipe=False  scientific=False


In [14]:
# Purpose: create the ingestion utilities used by optional indexing and workflow backfills.
from ingestion.index_builder import IndexBuilder
from ingestion.document_loader import DocumentLoader
from ingestion.chunking_pipeline import ChunkingPipeline

# Build the ingestion helper even when indexing is skipped. Later cells and
# the workflow constructor expect `builder` to exist for optional backfills.
builder = IndexBuilder(vector_store=vs)


In [15]:

# Check CUDA (GPU) availability and print details about the GPU and its memory.
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

CUDA available: True
GPU: NVIDIA A100-SXM4-80GB
Memory: 85.1 GB


<a id="9-industrial-indexing"></a>

## 9. Industrial indexing


In [16]:
# Purpose: optionally seed the industrial collection with starter troubleshooting examples.
if not REINDEX_INDUSTRIAL:
    print(
        "Skipping industrial starter samples (set REINDEX_INDUSTRIAL=True to rebuild)."
    )
else:
    # ── Industrial: Rockwell Automation / Allen-Bradley ──
    # Starter samples: only indexed when the industrial collection is empty.
    # Put real manuals under PROJECT_DIR/data/industrial/ and run the next cell.
    _industrial_count = vs.get_collection_stats("industrial")["document_count"]
    if _industrial_count == 0:
        sample_industrial = [
            """Fault Code F002 - Allen-Bradley PowerFlex 525 VFD: Auxiliary Input.
            Description: An external fault has been detected via the auxiliary input terminal.
            Possible Causes:
            1. External device connected to auxiliary input is reporting a fault condition
            2. Wiring issue on the auxiliary digital input terminal
            3. Incorrect parameter configuration for the auxiliary input function
            Troubleshooting Steps:
            1. Check the external device connected to the auxiliary input for faults
            2. Verify wiring connections on terminal block TB2
            3. Check parameter A070 [Fault Config 2] for correct auxiliary input configuration
            4. Inspect for damaged or loose wiring on the digital input terminals
            Resolution: Clear the external fault condition and reset the drive by cycling the Enable input or pressing the Stop/Reset key on the HIM.""",
    
            """Fault Code F004 - Allen-Bradley PowerFlex 525 VFD: Undervoltage.
            Description: The DC bus voltage has dropped below the undervoltage trip level.
            Possible Causes:
            1. Input power supply voltage too low or momentary power dip
            2. Incoming power supply fuses blown or circuit breaker tripped
            3. Loose connections on input power terminals R/L1, S/L2, T/L3
            4. DC bus capacitors degraded (check capacitor health indicator LED)
            Troubleshooting Steps:
            1. Measure incoming line voltage at drive terminals - must be within nameplate rating +/-10%
            2. Check all three phases for voltage balance (max 3% imbalance)
            3. Inspect and tighten input power terminal connections (torque to 1.4 Nm)
            4. Check parameter A531 [DC Bus Voltage] for current reading
            5. Review fault log in parameter A700-A706 for fault history and timestamps
            Resolution: Restore proper input voltage. If capacitors are degraded, replace the drive.
            SAFETY: Disconnect and lockout/tagout all power sources before inspecting terminals.
            Wait 5 minutes for DC bus capacitors to discharge before servicing.""",
    
            """Fault Code F005 - Allen-Bradley PowerFlex 525 VFD: Overvoltage.
            Description: The DC bus voltage has exceeded the overvoltage trip level.
            Possible Causes:
            1. Input line voltage exceeds drive nameplate rating
            2. Excessive regenerative energy from motor deceleration (overhauling load)
            3. Deceleration time too short for the load inertia
            4. Dynamic brake resistor circuit failure or incorrect sizing
            Troubleshooting Steps:
            1. Measure input voltage - must not exceed 528 VAC for 480V class drives
            2. Increase deceleration time in parameter A092 [Decel Time 1]
            3. Enable bus regulator: set parameter A540 [Bus Reg Mode] to option 1 (Enabled)
            4. If using dynamic braking, check DB resistor connections and resistance value
            5. Check parameter A531 [DC Bus Voltage] - nominal is ~650V for 480V input
            Resolution: Reduce input voltage or extend deceleration time. Add DB resistor for overhauling loads.""",
    
            """Allen-Bradley CompactLogix 5380 Controller - EtherNet/IP Configuration Guide.
            Model: 1769-L33ER CompactLogix 5380
            Prerequisites: RSLogix 5000 v32+ or Studio 5000 Logix Designer v32+
            Step 1: Create new project - select 1769-L33ER controller, revision 32+
            Step 2: Configure controller IP address via USB connection using BOOTP/DHCP tool
            Step 3: Add EtherNet/IP module in I/O Configuration tree
            Step 4: Right-click controller > Properties > set IP address (e.g., 192.168.1.10)
            Step 5: Add remote I/O devices - right-click EtherNet/IP > New Module
            Step 6: Configure Produced/Consumed tags for controller-to-controller communication
            Step 7: Download program and verify connection indicators:
              - OK LED: Solid green = running, Flashing green = program mode
              - ENET LED: Solid green = has connections, Flashing green = no connections
            Common Issue: If ENET LED is off, check IP config and network switch connection.
            Use RSLinx Classic > RSWho to verify controller appears on EtherNet/IP network.""",
    
            """Allen-Bradley PanelView Plus 7 HMI - Troubleshooting Communication Failures.
            Symptom: PanelView displays "Controller not found" or shows stale data.
            Possible Causes:
            1. Incorrect IP address or subnet mask configuration
            2. EtherNet/IP cable disconnected or faulty
            3. RSLinx Enterprise communication path misconfigured
            4. Controller is in Program mode instead of Run mode
            Troubleshooting Steps:
            1. On PanelView: press Terminal Settings > Networks > verify IP address
            2. Ping the controller IP from PanelView diagnostics screen
            3. In FactoryTalk View Studio: check Communication Setup > verify shortcut path
            4. Ensure controller and HMI are on same subnet (e.g., both 192.168.1.x/24)
            5. Check Ethernet cable and switch port LEDs for link activity
            6. Review controller mode - must be in Run or Remote Run for live data
            Resolution: Correct IP addressing and verify physical network connectivity.
            If using managed switch, verify VLANs are not isolating devices.""",
    
            """Preventive Maintenance Schedule - Allen-Bradley PowerFlex 755 Drive System.
            Weekly: Check drive status LEDs and HIM display for active alarms
            Monthly: Inspect cooling fans for proper operation, clean air filters
            Quarterly: Verify DC bus voltage (parameter 15 - Bus Voltage), check capacitor
            formation indicator, tighten all power and control terminal connections
            Semi-Annually: Thermal scan of power connections, verify ground fault monitoring,
            backup drive parameters using Connected Components Workbench (CCW)
            Annually: Full parameter backup, firmware version audit against Rockwell
            compatibility matrix, capacitor health test, clean power module heat sinks
            Every 5 Years: Replace cooling fans (recommended life), capacitor reformation
            MTBF: PowerFlex 755 rated at approximately 200,000 hours (28+ years) at 40C ambient.
            Critical Spares to Stock: Cooling fan assembly (KIT-F755FAN), control board fuse,
            fiber optic cables for multi-axis configurations.""",
        ]

        count = builder.index_industrial_texts(sample_industrial, source="rockwell_automation")
        print(f"✅ Indexed {count} industrial chunks")

    else:
        print(
            f"Industrial collection already has {_industrial_count} chunks; "
            "skipping starter samples. (Clear the DB or collection to re-seed.)"
        )


Skipping industrial starter samples (set REINDEX_INDUSTRIAL=True to rebuild).


### 9.1 Index manuals from Google Drive

Place `.pdf`, `.txt`, or `.md` files under `PROJECT_DIR/data/industrial/` on Drive, then run the next cell. Re-runs use `skip_existing=True` so unchanged files are not re-embedded.


In [17]:
# Purpose: optionally index industrial manuals stored in Google Drive.
if not REINDEX_INDUSTRIAL:
    print("Skipping §9.1 industrial indexing (set REINDEX_INDUSTRIAL=True to rebuild).")
else:
    import os
    from pathlib import Path

    _IND_DIR = os.path.join(PROJECT_DIR, "data", "industrial")
    _SUPPORTED = {".pdf", ".txt", ".md"}

    files: list[str] = []
    for root, _, fnames in os.walk(_IND_DIR):
        for fn in sorted(fnames):
            ext = Path(fn).suffix.lower()
            if ext in _SUPPORTED:
                files.append(os.path.join(root, fn))

    print(f"Industrial manuals dir: {_IND_DIR}")
    print(f"Found {len(files)} file(s) matching {sorted(_SUPPORTED)}")

    if not files:
        print(
            "No manuals to index. Add .pdf / .txt / .md under PROJECT_DIR/data/industrial on Drive, "
            "then re-run this cell."
        )
    else:
        total_chunks = 0
        total_added = 0
        for fpath in files:
            ext = Path(fpath).suffix.lower()
            try:
                if ext == ".pdf":
                    docs = builder.loader.load_pdf(fpath)
                else:
                    docs = builder.loader.load_text(fpath)
            except Exception as e:
                print(f"  SKIP {fpath}: {e}")
                continue
            stem = Path(fpath).stem
            for doc in docs:
                if doc.metadata.get("type") == "pdf":
                    pg = int(doc.metadata.get("page") or 0)
                    doc.metadata["id"] = f"{stem}_p{pg}"
                else:
                    doc.metadata["id"] = f"{stem}_txt"
            chunks = builder.chunker.chunk_documents(docs, domain="industrial")
            added = vs.add_documents("industrial", chunks, skip_existing=True)
            total_chunks += len(chunks)
            total_added += added
            print(
                f"  {Path(fpath).name}: {len(docs)} doc(s) -> {len(chunks)} chunk(s), "
                f"added {added} (skipped {len(chunks) - added})"
            )
        print(
            f"Done: {total_chunks} chunk(s) from files, {total_added} newly added "
            f"({total_chunks - total_added} already present)."
        )



Skipping §9.1 industrial indexing (set REINDEX_INDUSTRIAL=True to rebuild).


In [18]:
# Purpose: snapshot Chroma only after industrial data was re-indexed.
if REINDEX_INDUSTRIAL:
    snapshot_chroma("industrial")
else:
    print("Skipping industrial snapshot (nothing re-indexed this run).")


Skipping industrial snapshot (nothing re-indexed this run).


In [19]:
# Industrial index sanity check
print("Industrial collection size:", vs.get_collection_stats("industrial")["document_count"])

for q in (
    "PowerFlex 525 fault F004 undervoltage",
    "PanelView Controller not found",
):
    print(f"\n--- Semantic: {q!r} ---")
    for d in vs.search("industrial", q, k=3):
        m = d.metadata or {}
        print(
            f"  • source={m.get('source')!r}  page={m.get('page')}  "
            f"sim={m.get('similarity_score', 0):.3f}"
        )


Industrial collection size: 6150

--- Semantic: 'PowerFlex 525 fault F004 undervoltage' ---
  • source='rockwell_automation'  page=None  sim=0.934
  • source='rockwell_automation'  page=None  sim=0.902
  • source='rockwell_automation'  page=None  sim=0.879

--- Semantic: 'PanelView Controller not found' ---
  • source='rockwell_automation'  page=None  sim=0.893
  • source='rockwell_automation'  page=None  sim=0.800
  • source='750-TG101_-EN-P'  page=72  sim=0.799


<a id="10-recipe-indexing"></a>

## 10. Recipe indexing


### Incremental indexing

`index_recipes` now defaults to `skip_existing=True`: recipe IDs already in Chroma are skipped (no re-embedding). Re-runs after the first full index finish in seconds. Pass `skip_existing=False` to force a complete re-embed (e.g. after refreshing interaction data).


In [20]:
# Purpose: optionally build or refresh the Food.com recipe vector collection.
if not REINDEX_RECIPE:
    print("Skipping recipe indexing (set REINDEX_RECIPE=True to rebuild).")
else:
    # ── Recipe Domain (full corpus) ────────────────────────────────────────────
    # Expects the Food.com dump under data/recipe/:
    #   RAW_recipes.csv, RAW_interactions.csv
    # Set RECIPE_MAX_ROWS env var for a smoke test (e.g. 1000).
    import time as _t

    _recipe_dir   = os.path.join(PROJECT_DIR, "data", "recipe")
    _recipes_csv  = os.path.join(_recipe_dir, "RAW_recipes.csv")
    _interactions = os.path.join(_recipe_dir, "RAW_interactions.csv")

    # Backward-compat: older layout had RAW_recipes.csv directly under data/
    if not os.path.isfile(_recipes_csv):
        _legacy = os.path.join(PROJECT_DIR, "data", "RAW_recipes.csv")
        if os.path.isfile(_legacy):
            _recipes_csv = _legacy
            _interactions = os.path.join(PROJECT_DIR, "data", "RAW_interactions.csv")

    _max_rows_env = os.environ.get("RECIPE_MAX_ROWS", "").strip()
    _max_rows     = int(_max_rows_env) if _max_rows_env else None

    chunker = ChunkingPipeline()

    if os.path.isfile(_recipes_csv):
        _use_interactions = _interactions if os.path.isfile(_interactions) else None
        if _max_rows is None:
            print(f"Indexing ALL recipes (~231K) from {_recipes_csv} ...")
        else:
            print(f"Indexing up to {_max_rows} recipes from {_recipes_csv} ...")
        if _use_interactions:
            print(f"  + popularity enrichment from {_interactions}")
        else:
            print(f"  (no interactions CSV found; skipping popularity enrichment)")

        _t0 = _t.time()
        count = builder.index_recipes(
            _recipes_csv,
            max_rows=_max_rows,
            dedupe=True,                       # exact-hash dedupe on (name, ingredients)
            drop_junk=True,                    # filter implausible minutes / empty steps
            interactions_csv=_use_interactions,
            batch_size=1000,                   # embed + upsert 1K recipes per round-trip
        )
        _elapsed = _t.time() - _t0
        print(f"✅ Indexed {count:,} recipe chunks in {_elapsed/60:.1f} min "
              f"({count / max(_elapsed, 1e-9):.0f} chunks/sec)")
    else:
        print(f"⚠️  RAW_recipes.csv not found at {_recipes_csv} — using demo recipes only.")
        print("   Place the Food.com dump under data/recipe/ for full-corpus indexing.")
        sample_recipes = [
            """Recipe: Classic Chocolate Chip Cookies
            Prep Time: 45 minutes. Ingredients: 2 1/4 cups flour, 1 tsp baking soda,
            1 tsp salt, 1 cup butter softened, 3/4 cup sugar, 3/4 cup brown sugar,
            2 large eggs, 2 tsp vanilla, 2 cups chocolate chips.
            Egg Substitutes: Use 1/4 cup unsweetened applesauce per egg,
            or 1 mashed banana per egg, or 3 tbsp aquafaba per egg.""",

            """Recipe: Quick Chicken Stir-Fry
            Prep Time: 20 minutes. Ingredients: 1 lb chicken breast, 2 bell peppers,
            1 cup rice, 3 tbsp soy sauce, 1 tbsp sesame oil, 2 cloves garlic, ginger.
            Steps: Cook rice. Slice chicken. Heat oil in wok over high heat.
            Stir-fry chicken 5-6 min. Add vegetables and garlic. Add soy sauce.
            Nutrition: ~450 calories per serving, 35g protein.""",

            """Recipe: Homemade Margherita Pizza
            Prep Time: 90 minutes (including dough rise). Ingredients: 3 cups flour,
            1 packet yeast, 1 cup warm water, 2 tbsp olive oil, 1 tsp salt, 1 tsp sugar,
            1 cup crushed San Marzano tomatoes, 8 oz fresh mozzarella, fresh basil.
            Steps: Mix dough, let rise 1 hour. Preheat oven to 475F. Stretch dough,
            add sauce and cheese. Bake 12-15 min until crust is golden.
            Gluten-Free Alternative: Substitute 3 cups gluten-free flour blend plus
            1 tsp xanthan gum for regular flour.""",
        ]
        recipe_docs = DocumentLoader.from_texts(sample_recipes, "recipe", "demo_recipes")
        recipe_chunks = chunker.chunk_documents(recipe_docs, domain="recipe")
        count = vs.add_documents("recipe", recipe_chunks)
        print(f"✅ Indexed {count} demo recipe chunks")



Skipping recipe indexing (set REINDEX_RECIPE=True to rebuild).


In [21]:
# Purpose: snapshot Chroma only after recipe data was re-indexed.
if REINDEX_RECIPE:
    snapshot_chroma("recipe")
else:
    print("Skipping recipe snapshot (nothing re-indexed this run).")


Skipping recipe snapshot (nothing re-indexed this run).


In [22]:
# Purpose: debug which recipe indexing implementation is active in the kernel.
import os, inspect
import ingestion.index_builder as _ib

# Where is the module actually loading from?
print("Module file:", _ib.__file__)

# Does the method have 'dedupe' as a parameter?
sig = inspect.signature(_ib.IndexBuilder.index_recipes)
print("Params     :", list(sig.parameters.keys()))

# Shows the actual source of index_recipes in the running kernel
print(inspect.getsource(_ib.IndexBuilder.index_recipes)[:400])


Module file: /content/AI574/ingestion/index_builder.py
Params     : ['self', 'csv_path', 'max_rows', 'dedupe', 'drop_junk', 'interactions_csv', 'batch_size', 'skip_existing']
    def index_recipes(
        self,
        csv_path: str,
        max_rows: Optional[int] = None,
        *,
        dedupe: bool = True,
        drop_junk: bool = True,
        interactions_csv: Optional[str] = None,
        batch_size: int = 1000,
        skip_existing: bool = True,
    ) -> int:
        """
        Index Food.com recipes from CSV.

        Parameters
        ----------
      


In [23]:
# ── Recipe Index Sanity Check ────────────────────────────────────────────
# Confirms the new metadata is queryable end-to-end:
#   - unfiltered semantic retrieval
#   - metadata-filtered retrieval (quick + vegetarian, high popularity)
#   - metadata distribution snapshot
from collections import Counter

print("📊 Recipe collection size:", vs.get_collection_stats("recipe")["document_count"])

# 1) plain semantic search
print("\n--- Semantic: 'quick weeknight pasta with garlic' ---")
for d in vs.search("recipe", "quick weeknight pasta with garlic", k=3):
    m = d.metadata
    print(f"  • {m.get('name')!r}  ({m.get('minutes_bucket')}, "
          f"{m.get('review_count',0)}★x{m.get('avg_rating',0):.1f}, "
          f"sim={m.get('similarity_score', 0):.3f})")

# 2) filtered search — vegetarian AND quick AND popular
#    Chroma where: combine with $and; tags use $contains on the |tag| sentinel.
print("\n--- Filtered: vegetarian + quick + popular (≥10 reviews) ---")
where = {
    "$and": [
        {"is_vegetarian": {"$eq": True}},
        {"is_quick":      {"$eq": True}},
        {"review_count":  {"$gte": 10}},
    ]
}
try:
    hits = vs.search("recipe", "one-pot vegetable dinner", k=5, where=where)
    for d in hits:
        m = d.metadata
        print(f"  • {m.get('name')!r}  {m.get('minutes')}min, "
              f"{m.get('review_count')}x{m.get('avg_rating'):.1f}, "
              f"tier={m.get('popularity_tier')}")
except Exception as e:
    print(f"  (filtered search not supported by this Chroma version: {e})")

# 3) metadata distribution — just the first ~2K to stay fast
print("\n--- Metadata distribution (sample of 2K chunks) ---")
coll = vs._collections["recipe"]
sample = coll.get(limit=2000, include=["metadatas"])
metas = sample.get("metadatas", []) or []
if metas:
    tiers   = Counter(m.get("popularity_tier", "n/a") for m in metas)
    buckets = Counter(m.get("minutes_bucket", "n/a") for m in metas)
    veg     = sum(1 for m in metas if m.get("is_vegetarian"))
    quick   = sum(1 for m in metas if m.get("is_quick"))
    gf      = sum(1 for m in metas if m.get("is_gluten_free"))
    print(f"  popularity: {dict(tiers)}")
    print(f"  prep time:  {dict(buckets)}")
    print(f"  flags: vegetarian={veg}, quick={quick}, gluten-free={gf} / {len(metas)} sampled")


📊 Recipe collection size: 231311

--- Semantic: 'quick weeknight pasta with garlic' ---
  • 'parmesan garlic pasta'  (quick, 1★x5.0, sim=0.938)
  • 'quick pasta for one'  (quick, 13★x4.4, sim=0.923)
  • 'creamy garlic pasta'  (medium, 8★x3.8, sim=0.922)

--- Filtered: vegetarian + quick + popular (≥10 reviews) ---
  • 'sauteed peas with mushrooms and garlic'  20min, 16x4.8, tier=medium
  • 'green beans and carrots sauteed in butter and garlic'  18min, 18x4.8, tier=medium
  • 'broccoli and cauliflower saute'  20min, 23x4.3, tier=medium
  • 'thick   chunky tomato sauce with veggies  crock pot'  26min, 12x5.0, tier=medium
  • 'brown rice and vegetable saute'  20min, 19x4.5, tier=medium

--- Metadata distribution (sample of 2K chunks) ---
  popularity: {'low': 1781, 'medium': 202, 'high': 17}
  prep time:  {'medium': 644, 'long': 571, 'quick': 785}
  flags: vegetarian=300, quick=785, gluten-free=41 / 2000 sampled


<a id="11-scientific-indexing"></a>

## 11. Scientific indexing


### Scientific indexing — Cornell-University/arxiv
Downloads the ArXiv metadata dump once, caches on Drive, and bulk-indexes cs.* papers from 2020+. ScientificAgent still backfills live via the ArXiv API for anything the local index misses.


In [24]:
# Purpose: optionally build or refresh the ArXiv scientific vector collection.
if not REINDEX_SCIENTIFIC:
    print("Skipping scientific arxiv indexing (set REINDEX_SCIENTIFIC=True to rebuild).")
else:
    # ── Scientific Domain: Cornell-University/arxiv bulk index ─────────────────
    # Strategy:
    #   1. Pull Cornell-University/arxiv via kagglehub (uses creds synced in cell 8)
    #   2. Cache the NDJSON under PROJECT_DIR/data/scientific/ on Drive so it
    #      survives runtime restarts (no re-download).
    #   3. Bulk-index cs.* papers from 2020+ via IndexBuilder.index_arxiv_dump.
    #   4. For misses at query time, ScientificAgent already backfills via the
    #      live ArXiv API — this cell only seeds the local index.
    #
    # Knobs (env):
    #   SCI_MAX_ROWS   cap kept papers (smoke test)
    #   SCI_CATEGORIES space-separated category prefixes (default: "cs.")
    #   SCI_YEAR_MIN   default 2020
    #   SCI_FORCE=1    re-embed everything, ignore skip_existing
    import os, shutil, time as _t
    from pathlib import Path

    # Kaggle dataset → local cache location on Drive
    _SCI_DIR   = Path(PROJECT_DIR) / "data" / "scientific"
    _SCI_FILE  = _SCI_DIR / "arxiv-metadata-oai-snapshot.json"
    _SCI_DIR.mkdir(parents=True, exist_ok=True)

    if not _SCI_FILE.is_file():
        print("Downloading Cornell-University/arxiv via kagglehub (~4 GB, one-time)…")
        import kagglehub
        _cache_dir = kagglehub.dataset_download("Cornell-University/arxiv")
        _cache_path = Path(_cache_dir)
        # kagglehub returns the directory; find the NDJSON inside
        _candidates = list(_cache_path.glob("arxiv-metadata-oai-snapshot.json")) + \
                      list(_cache_path.rglob("arxiv-metadata-oai-snapshot.json"))
        if not _candidates:
            raise FileNotFoundError(
                f"kagglehub cache at {_cache_path} missing arxiv-metadata-oai-snapshot.json"
            )
        _src = _candidates[0]
        print(f"  copying {_src} → {_SCI_FILE}")
        # copy rather than move so kagglehub's cache stays intact for resumes
        shutil.copy2(_src, _SCI_FILE)
    else:
        print(f"Using cached dump: {_SCI_FILE} "
              f"({_SCI_FILE.stat().st_size / 1e9:.2f} GB)")

    # ── Filter knobs ────────────────────────────────────────────────────────────
    _sci_categories = tuple(
        os.environ.get("SCI_CATEGORIES", "cs.").split()
    ) or ("cs.",)
    _sci_year_min   = int(os.environ.get("SCI_YEAR_MIN", "2020"))
    _sci_max_env    = os.environ.get("SCI_MAX_ROWS", "").strip()
    _sci_max_rows   = int(_sci_max_env) if _sci_max_env else None
    _sci_skip       = os.environ.get("SCI_FORCE", "") != "1"

    print(f"Indexing arxiv papers — categories={list(_sci_categories)}, "
          f"year_min={_sci_year_min}, max_rows={_sci_max_rows}, "
          f"skip_existing={_sci_skip}")

    _t0 = _t.time()
    count = builder.index_arxiv_dump(
        str(_SCI_FILE),
        max_rows=_sci_max_rows,
        categories=_sci_categories,
        year_min=_sci_year_min,
        batch_size=1000,
        skip_existing=_sci_skip,
    )
    _elapsed = _t.time() - _t0
    print(f"✅ Indexed {count:,} new arxiv chunks in {_elapsed/60:.1f} min "
          f"({count / max(_elapsed, 1e-9):.0f} chunks/sec)")

    # Collection status
    print("\n📊 Index status:")
    for stat in vs.get_all_stats():
        print(f"  {stat['domain']:>12}: {stat['document_count']:>8,} chunks")



Skipping scientific arxiv indexing (set REINDEX_SCIENTIFIC=True to rebuild).


In [25]:
# Purpose: snapshot Chroma only after scientific data was re-indexed.
if REINDEX_SCIENTIFIC:
    snapshot_chroma("scientific")
else:
    print("Skipping scientific snapshot (nothing re-indexed this run).")


Skipping scientific snapshot (nothing re-indexed this run).


In [26]:
# ── Scientific Index Sanity Check ────────────────────────────────────────
# Same pattern as the recipe sanity check: semantic, filtered, distribution.
from collections import Counter

print("📊 Scientific collection size:", vs.get_collection_stats("scientific")["document_count"])

# 1) plain semantic search
print("\n--- Semantic: 'retrieval-augmented generation for question answering' ---")
for d in vs.search("scientific", "retrieval-augmented generation for question answering", k=3):
    m = d.metadata
    print(f"  • {m.get('title', '')[:70]!r}")
    print(f"      {m.get('primary_category')}  {m.get('year')}-{m.get('month'):02d}  "
          f"authors={m.get('authors_short', '')[:40]!r}  "
          f"sim={m.get('similarity_score', 0):.3f}")

# 2) filtered search — NLP papers with DOI
print("\n--- Filtered: cs.CL (NLP) + has DOI ---")
where = {
    "$and": [
        {"is_nlp":  {"$eq": True}},
        {"has_doi": {"$eq": True}},
    ]
}
try:
    hits = vs.search("scientific", "transformer language models", k=5, where=where)
    for d in hits:
        m = d.metadata
        print(f"  • {m.get('title', '')[:70]!r}  ({m.get('primary_category')}, "
              f"{m.get('year')})  doi={m.get('doi', '')[:40]}")
except Exception as e:
    print(f"  (filtered search not supported: {e})")

# 3) metadata distribution — first ~2K chunks
print("\n--- Metadata distribution (sample of 2K chunks) ---")
coll = vs._collections["scientific"]
sample = coll.get(limit=2000, include=["metadatas"])
metas = sample.get("metadatas", []) or []
if metas:
    years = Counter(m.get("year", 0) for m in metas)
    prims = Counter(m.get("primary_category", "?") for m in metas).most_common(8)
    ml    = sum(1 for m in metas if m.get("is_ml"))
    nlp   = sum(1 for m in metas if m.get("is_nlp"))
    cv    = sum(1 for m in metas if m.get("is_cv"))
    doi   = sum(1 for m in metas if m.get("has_doi"))
    comm  = sum(1 for m in metas if m.get("has_comments"))
    print(f"  years:        {dict(sorted(years.items()))}")
    print(f"  top primary:  {prims}")
    print(f"  flags: ML={ml}, NLP={nlp}, CV={cv}, has_doi={doi}, has_comments={comm} / {len(metas)}")


📊 Scientific collection size: 686968

--- Semantic: 'retrieval-augmented generation for question answering' ---
  • 'Retrieval Augmented Generation for Domain-specific Question Answering'
      cs.CL  2024-05  authors='Sanat Sharma, David Seunghyun Yoon, Fran'  sim=0.923
  • 'Generation-Augmented Retrieval for Open-domain Question Answering'
      cs.CL  2021-08  authors='Yuning Mao, Pengcheng He, Xiaodong Liu, '  sim=0.912
  • 'Question Decomposition for Retrieval-Augmented Generation'
      cs.CL  2025-07  authors='Paul J. L. Ammann, Jonas Golde, Alan Akb'  sim=0.910

--- Filtered: cs.CL (NLP) + has DOI ---
  • 'Transformer Grammars: Augmenting Transformer Language Models with   Sy'  (cs.CL, 2022)  doi=10.1162/tacl_a_00526
  • 'Tree-Planted Transformers: Unidirectional Transformer Language Models '  (cs.CL, 2025)  doi=10.18653/v1/2024.findings-acl.303
  • 'Fine-Tuning Transformers: Vocabulary Transfer'  (cs.CL, 2024)  doi=10.1016/j.artint.2023.103860
  • 'Probabilistic Transformer: A

<a id="12-build-workflow"></a>

## 12. Build workflow


In [27]:
# Purpose: compile the LangGraph workflow used by all demos and evaluations.
import importlib
import config.prompts
import rag_core.response_generator
import rag_core.crag_pipeline
import orchestration.workflow_graph
import orchestration.state_schema

# Reload local modules so this notebook picks up recent repo edits without a
# full Colab runtime restart. This is useful while iterating on prompts/graph code.
importlib.reload(config.prompts)
importlib.reload(rag_core.response_generator)
importlib.reload(rag_core.crag_pipeline)
importlib.reload(orchestration.state_schema)
importlib.reload(orchestration.workflow_graph)
from orchestration.workflow_graph import build_workflow, run_query

# Fail fast with a helpful message if a user skipped setup/indexing cells.
try:
    llm, vs, builder
except NameError:
    raise RuntimeError(
        "Run the cells above first so 'llm', 'vs', and 'builder' exist: "
        "load LLM, create embedder & vector store, run IndexBuilder and index data."
    )

# Build the LangGraph state machine. The compiled workflow contains supervisor
# routing, domain agents, synthesis, clarification, and fallback paths.
workflow = build_workflow(llm=llm, vector_store=vs, index_builder=builder)
print("✅ Workflow compiled!")



✅ Workflow compiled!


<a id="live-demonstration-planned-queries"></a>

### Demo Queries

The demo covers five behaviors: industrial, recipe, scientific, synthesis, and fallback. The output shows route, sources, response, and timing.


<a id="125-live-five-query-demo-path"></a>

## 12.5 Five-Query Demo

This cell runs the compact demo used for submission: one industrial query, one recipe query, one scientific query, one synthesis query, and one fallback query.


In [ ]:
# Live demo query set: one query per key behavior.
#
# These five rows are the shortest path through the system for a recorded demo:
#   1. normal single-domain industrial troubleshooting
#   2. normal single-domain recipe assistance
#   3. normal single-domain scientific summarization
#   4. cross-domain synthesis over recipe + scientific evidence
#   5. out-of-scope fallback through web search
#
# Keep this list small and stable so the video and the results table in §21.6
# talk about the exact same test cases.
LIVE_DEMO_QUERIES = [
    {
        "id": 1,
        "label": "Industrial troubleshooting",
        "query": "My PowerFlex 525 drive shows fault F004. What should I check?",
        "expected": "industrial",
        "options": {},
    },
    {
        "id": 2,
        "label": "Recipe substitution",
        "query": "What can I substitute for eggs in pancakes?",
        "expected": "recipe",
        "options": {},
    },
    {
        "id": 3,
        "label": "Scientific summarization",
        "query": "Summarize recent research on retrieval-augmented generation for NLP.",
        "expected": "scientific",
        "options": {},
    },
    {
        "id": 4,
        "label": "Cross-domain synthesis",
        "query": "I am preparing a presentation on retrieval-augmented generation in NLP. Summarize the recent research, and suggest a quick dinner recipe I can make before presenting.",
        "expected": "synthesis",
        "options": {"enable_synthesis": True, "k": 3},
        "demo_runner": "direct_synthesis",
        "synthesis_domains": ["recipe", "scientific"],
    },
    {
        "id": 5,
        "label": "Out-of-scope fallback",
        "query": "Who won the 2024 Super Bowl and what was the final score?",
        "expected": "fallback",
        "options": {},
    },
]


DISPLAY_FULL_DEMO_RESPONSES = True


def display_result(result, case=None):
    """Pretty-print workflow result with optional demo metadata and CRAG timing breakdown.

    Same spirit as Appendix ``display_result`` in ``Multi_Domain_Agent.ipynb``: header,
    full response (or truncated when DISPLAY_FULL_DEMO_RESPONSES is False), then optional
    stage timing bars.
    """
    domain = result.get("domain", "?")
    conf = float(result.get("confidence") or 0)
    status = result.get("status", "?")
    esc = result.get("escalated", False)
    srcs = result.get("sources", []) or []
    timing = result.get("timing", {}) or {}

    sep = "=" * 60
    print(f"\n{sep}")
    if case is not None:
        print(f"  Query {case['id']}: {case['label']}")
        print(f"  Expected route : {case['expected']}")
    print(f"  Domain: {str(domain):<14} Confidence: {conf:.2f}")
    print(f"  Routing source: {result.get('routing_source') or '(n/a)'}")
    print(f"  Status: {str(status):<14} Escalated:  {esc}")
    print(f"  Sources: {len(srcs)}")
    print(f"  Total latency  : {float(timing.get('total_s', 0) or 0):.2f}s")
    if result.get("synthesis_domains_used"):
        print(f"  Synthesis used : {result.get('synthesis_domains_used')}")
    if result.get("web_search_provider"):
        print(f"  Web provider   : {result.get('web_search_provider')}")
    print(sep)

    response = (result.get("response") or "").strip()
    if DISPLAY_FULL_DEMO_RESPONSES:
        body = response
    else:
        body = response[:700] + ("..." if len(response) > 700 else "")
    print(f"\n{body if body else '[empty response]'}\n")

    t = result.get("timing")
    if not t:
        print("  (no timing breakdown — optional stages may be missing)\n")
        return

    total = float(t.get("total_s", 0) or 0.001)
    sup = float(t.get("supervisor_s", 0) or 0)
    agent = float(t.get("agent_s", 0) or 0)
    c = t.get("crag") or {}

    retrieve = float(c.get("retrieve_s", 0) or 0)
    grade = float(c.get("grade_s", 0) or 0)
    rewrite = float(c.get("rewrite_s", 0) or 0)
    generate = float(c.get("generate_s", 0) or 0)
    validate = float(c.get("validate_s", 0) or 0)
    overhead = max(0.0, agent - (retrieve + grade + rewrite + generate + validate))

    steps = [
        ("Supervisor", sup),
        ("  Retrieve", retrieve),
        ("  Grade", grade),
        ("  Rewrite", rewrite),
        ("  Generate", generate),
        ("  Validate", validate),
        ("  Overhead", overhead),
    ]

    BAR = 25
    line = "-" * 60
    print(line)
    print(f"  TIMING BREAKDOWN  (total {total:.1f}s / {total/60:.2f} min)")
    print(line)
    for label, s in steps:
        if s < 0.01 and label == "  Rewrite":
            continue
        pct = (s / total) * 100 if total else 0.0
        fill = int(BAR * s / total) if total else 0
        bar = "#" * fill + "-" * (BAR - fill)
        print(f"  {label:<12} {bar} {s:5.1f}s  ({pct:4.1f}%)")
    print(line)
    print(f"  {'Total':<12} {'':>{BAR}} {total:5.1f}s")
    print(line)
    print()


def display_live_demo_result(case, result):
    """Compact output for the presentation demo."""
    display_result(result, case=case)


def run_direct_synthesis_demo(case, *, mode="best_quality"):
    """Run the synthesis agent directly for a deterministic cross-domain demo."""
    # The normal workflow only upgrades `clarify -> synthesis` when the
    # supervisor returns two concrete specialist candidate domains. In live
    # demos, LLM routing can instead ask a clarification question. Calling the
    # SynthesisAgent directly makes the cross-domain feature deterministic.
    from rag_core.crag_pipeline import CRAGPipeline
    from agents.synthesis_agent import SynthesisAgent

    options = case.get("options") or {}
    domains = case.get("synthesis_domains") or ["recipe", "scientific"]

    # In fast mode, skip rewrite retries inside each per-domain CRAG call to
    # keep presentation latency manageable. Full-quality mode uses defaults.
    max_rewrites = 0 if mode == "fast_interactive" else options.get("max_rewrite_attempts")

    # Reuse the same LLM/vector store as the main workflow, but instantiate a
    # small synthesis-only path so this demo does not depend on supervisor
    # ambiguity thresholds.
    synth = SynthesisAgent(CRAGPipeline(llm, vs), llm)
    synth_result = synth.handle(
        query=case["query"],
        domains=domains,
        k=options.get("k", 3),
        max_rewrite_attempts=max_rewrites,
    )
    # Convert the SynthesisAgent payload into the same result shape used by
    # `display_result` / `display_live_demo_result`, so all five demo rows print consistently.
    return {
        "domain": "synthesis",
        "confidence": 1.0,
        "routing_source": "direct_synthesis_demo",
        "response": synth_result.get("response", ""),
        "sources": synth_result.get("sources", []),
        "timing": synth_result.get("timing", {}),
        "escalated": synth_result.get("escalated", False),
        "synthesis_domains_used": synth_result.get("domains_used", domains),
        "synthesis_per_domain": synth_result.get("per_domain_results", {}),
    }


def run_demo_case(case, *, mode="best_quality"):
    """Dispatch one live-demo case through the appropriate demo runner."""
    # Most rows use the normal LangGraph workflow. The synthesis row uses the
    # deterministic direct synthesis runner described above.
    if case.get("demo_runner") == "direct_synthesis":
        return run_direct_synthesis_demo(case, mode=mode)
    return run_query(
        workflow,
        case["query"],
        mode=mode,
        options=case.get("options") or {},
    )


def run_live_demo(cases=LIVE_DEMO_QUERIES, *, mode="best_quality"):
    """
    Run the five-query live demo.

    Use mode="best_quality" for full-quality defaults. Use mode="fast_interactive"
    if you need a shorter live presentation and are comfortable skipping some validation.
    """
    rows = []
    for case in cases:
        # Run one case at a time so a failure or slow query is easy to isolate
        # during the presentation. Each result is also summarized into `rows`
        # for a compact report table.
        result = run_demo_case(case, mode=mode)
        display_live_demo_result(case, result)
        rows.append({
            "id": case["id"],
            "label": case["label"],
            "expected": case["expected"],
            "actual": result.get("domain"),
            "routing_source": result.get("routing_source"),
            "confidence": result.get("confidence"),
            "escalated": result.get("escalated", False),
            "sources": len(result.get("sources", []) or []),
            "latency_s": (result.get("timing") or {}).get("total_s", 0),
        })
    return rows


# Submission default: skip 5 long queries on a fresh 'Run all' (set True for full tour).
RUN_FIVE_QUERY_LIVE_DEMO = False
if RUN_FIVE_QUERY_LIVE_DEMO:
    live_demo_rows = run_live_demo(mode="best_quality")
else:
    live_demo_rows = None
    print("Skipping automatic five-query run. Set RUN_FIVE_QUERY_LIVE_DEMO = False to run all five.")


In [29]:
# Live-demo control cell.
# Change DEMO_QUERY_ID from 1 to 5 and rerun this cell during the presentation.
DEMO_QUERY_ID = 1
DEMO_MODE = "best_quality"  # set to "fast_interactive" for a shorter demo

case = next(c for c in LIVE_DEMO_QUERIES if c["id"] == DEMO_QUERY_ID)
result = run_demo_case(case, mode=DEMO_MODE)
display_live_demo_result(case, result)



  Query 1: Industrial troubleshooting
  Expected route : industrial
  Domain: industrial     Confidence: 1.00
  Routing source: hybrid_classifier
  Status: complete       Escalated:  False
  Sources: 1
  Total latency  : 15.19s

Your PowerFlex 525 drive showing fault F004 indicates an Undervoltage condition where the DC bus voltage has dropped below the undervoltage trip level (Document 1).

**SAFETY WARNING:**
**Disconnect and lockout/tagout all power sources before inspecting terminals. Wait 5 minutes for DC bus capacitors to discharge before servicing.** (Document 1)

Here's what you should check, following the troubleshooting steps for Fault Code F004 (Document 1):

**Possible Causes:**
*   Input power supply voltage too low or momentary power dip.
*   Incoming power supply fuses blown or circuit breaker tripped.
*   Loose connections on input power terminals R/L1, S/L2, T/L3.
*   DC bus capacitors degraded (check capacitor health indicator LED).

**Troubleshooting Steps:**

1.  *

---

Sections §13–§19 are omitted from this clean submission notebook. The full appendix notebook contains the extra routing and diagnostic experiments.


<a id="20-basic-rag-baseline"></a>

## 20. Basic RAG Baseline

`basic_rag(query, domain=...)` retrieves from one selected domain and generates an answer. It has no routing, grading, rewriting, or hallucination validation.

The full quantitative baseline is `monolithic_rag` in §21.7, which pools all three domains and compares directly against `multi_agent_crag`.


In [ ]:
# Basic RAG baseline: retrieve top-k documents, then generate directly.
# This uses the same vector store and LLM as CRAG, but disables grading,
# query rewriting, and hallucination validation.

import time
from langchain_core.messages import HumanMessage


def basic_rag(query, domain="recipe", k=5, max_context_chars=10_000):
    """Run a minimal RAG baseline for comparison with CRAG."""
    # This intentionally keeps the baseline simple: retrieve top-k chunks from
    # one domain, then ask the LLM to answer from that context. There is no
    # supervisor routing, document grading, query rewriting, or hallucination
    # check. That makes the comparison with full CRAG easy to interpret.
    t0 = time.perf_counter()

    # Step 1: semantic retrieval from a single specified Chroma collection.
    # Any bad retrieval here goes straight into generation, unlike CRAG.
    retrieve_t0 = time.perf_counter()
    docs = vs.search(domain, query, k=k)
    retrieve_s = time.perf_counter() - retrieve_t0

    if not docs:
        return {
            "query": query,
            "domain": domain,
            "response": "No documents were retrieved for this query.",
            "sources": [],
            "timing": {"retrieve_s": retrieve_s, "generate_s": 0.0, "total_s": time.perf_counter() - t0},
        }

    # Step 2: build a bounded context window. The cap prevents very long
    # retrieved chunks from making the prompt too large for the hosted model.
    context_parts = []
    total_chars = 0
    per_doc_budget = max(500, max_context_chars // max(len(docs), 1))
    for i, doc in enumerate(docs, 1):
        source = doc.metadata.get("source") or doc.metadata.get("name") or doc.metadata.get("title") or "unknown"
        chunk = doc.page_content[:per_doc_budget]
        part = f"[Document {i} | Source: {source}]\n{chunk}"
        context_parts.append(part)
        total_chars += len(part)
        if total_chars >= max_context_chars:
            break

    context = "\n\n---\n\n".join(context_parts)
    # Step 3: one generation call. The prompt explicitly says to use only the
    # retrieved context, but Basic RAG does not verify whether the model obeyed.
    prompt = f"""You are a helpful assistant.
Answer the user query using only the retrieved context below.
If the context is insufficient, say so clearly.

Retrieved Context:
{context}

User Query: {query}
"""

    generate_t0 = time.perf_counter()
    response = llm.invoke([HumanMessage(content=prompt)]).content.strip()
    generate_s = time.perf_counter() - generate_t0

    # Preserve source metadata so the baseline can be compared against CRAG on
    # source count and retrieval quality, not just answer text.
    sources = [
        {
            "source": doc.metadata.get("source", "unknown"),
            "name": doc.metadata.get("name", ""),
            "title": doc.metadata.get("title", ""),
            "chunk_index": doc.metadata.get("chunk_index", -1),
            "similarity_score": doc.metadata.get("similarity_score", 0.0),
        }
        for doc in docs
    ]

    return {
        "query": query,
        "domain": domain,
        "response": response,
        "sources": sources,
        "timing": {
            "retrieve_s": retrieve_s,
            "generate_s": generate_s,
            "total_s": time.perf_counter() - t0,
        },
    }


def display_basic_rag_result(result):
    """Pretty-print Basic RAG output."""
    timing = result.get("timing", {})
    sources = result.get("sources", [])
    print(f"\n{'=' * 60}")
    print(f"  BASELINE: Basic RAG")
    print(f"  Domain: {result.get('domain', '?'):<14} Sources: {len(sources)}")
    print(f"  Retrieve: {timing.get('retrieve_s', 0):.2f}s | Generate: {timing.get('generate_s', 0):.2f}s | Total: {timing.get('total_s', 0):.2f}s")
    print(f"{'=' * 60}\n")
    print(result.get("response", ""))
    print("\nSources:")
    for i, src in enumerate(sources, 1):
        label = src.get("title") or src.get("name") or src.get("source") or "unknown"
        print(f"  [{i}] sim={src.get('similarity_score', 0):.3f}  {label}")


# Example baseline query. Compare this with the CRAG recipe demo above.
basic_result = basic_rag("Recipe for pancake?", domain="recipe", k=5)
display_basic_rag_result(basic_result)


<a id="21-baselines-vs-crag-ablation-comparison"></a>

## 21. Baselines vs CRAG

| Pipeline | Routing | Retrieval | CRAG checks | Purpose |
|---|:---:|:---:|:---:|---|
| Direct LLM | No | No | No | Ungrounded baseline. |
| Basic RAG | No | Yes | No | Retrieval-only baseline. |
| Routed Basic RAG | Yes | Yes | No | Tests routing without correction. |
| Full CRAG | Yes | Yes | Yes | Main system with grading, rewrite, and validation. |

The formal result in §21.7 compares the main system, `multi_agent_crag`, with the pooled Basic RAG-style baseline, `monolithic_rag`.


<a id="215-learned-router-and-grader-extension"></a>

## 21.5 Optional Learned Router / Grader

This is an optional speed extension, not the main RAG baseline comparison.

| Decision point | Learned model | Result | Latency |
|---|---|---|---|
| Document grading | MiniLM cross-encoder | Accuracy 0.899, binary F1 0.882 | 1482.81 ms/pair -> 7.21 ms/pair |
| Domain routing | DistilBERT classifier | Accuracy 1.000, macro-F1 1.000 | 1088.57 ms/query -> 19.39 ms/query |

The learned models replace expensive LLM routing/grading calls inside CRAG. They use silver labels, so they are best reported as deployment optimizations.

### Optional learned-model cells

These cells stay disabled by default. Turn on the flags only to retrain, re-evaluate, or smoke-test the learned router/grader.

In [31]:
# Fine-tuning artifact setup merged from Finetune_Grader_and_Router.ipynb.
# This cell only configures paths/imports. It does not train or evaluate models.
import os
import json
import time
import shutil
from pathlib import Path
from typing import Dict, List

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

# Keep expensive operations opt-in for the clean submission notebook.
RUN_LEARNED_MODEL_TRAINING = False
RUN_LEARNED_MODEL_EVALUATION = False
USE_LEARNED_GRADER = False
USE_LEARNED_ROUTER = False

ROUTER_LABELS = ["industrial", "recipe", "scientific", "clarify", "fallback"]

# Prefer the same Drive-backed artifact root used by the fine-tuning notebook.
FINETUNE_ARTIFACT_ROOT = Path("/content/drive/MyDrive/AI574/finetune_artifacts")
if not FINETUNE_ARTIFACT_ROOT.exists():
    # Local fallback for non-Colab runs.
    _project_root = Path(globals().get("GIT_REPO", "/content/AI574"))
    if not _project_root.exists():
        _project_root = Path.cwd()
    FINETUNE_ARTIFACT_ROOT = _project_root / "data" / "finetune_artifacts"

FINETUNE_DATA_DIR = FINETUNE_ARTIFACT_ROOT / "data" / "finetune"
FINETUNE_MODEL_DIR = FINETUNE_ARTIFACT_ROOT / "models"
FINETUNE_DATA_DIR.mkdir(parents=True, exist_ok=True)
FINETUNE_MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Preserve names from the fine-tuning notebook for copied cells below.
DATA_DIR = FINETUNE_DATA_DIR
MODEL_DIR = FINETUNE_MODEL_DIR
FORCE_RETRAIN_GRADER = False
FORCE_RETRAIN_ROUTER = False


def _load_jsonl(path: Path) -> List[dict]:
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def _model_artifact_exists(path: Path) -> bool:
    return path.exists() and path.is_dir() and any(path.iterdir())


def _data_artifact_exists(name: str) -> bool:
    path = DATA_DIR / name
    return path.exists() and path.stat().st_size > 0


def _resolve_saved_model_dir(model_name: str) -> Path:
    candidates = [
        MODEL_DIR / model_name,
        Path("/content/AI574/models") / model_name,
        Path.cwd() / "models" / model_name,
    ]
    for candidate in candidates:
        if candidate.exists() and candidate.is_dir() and any(candidate.iterdir()):
            return candidate
    raise FileNotFoundError(
        f"Could not find trained model '{model_name}'. Expected {MODEL_DIR / model_name}. "
        "Run the fine-tuning notebook or enable RUN_LEARNED_MODEL_TRAINING."
    )

print("Fine-tuning DATA_DIR:", DATA_DIR)
print("Fine-tuning MODEL_DIR:", MODEL_DIR)
print("Cached grader data:", _data_artifact_exists("grader_gold.jsonl"))
print("Cached router data:", _data_artifact_exists("router_gold.jsonl"))
print("Cached grader model:", _model_artifact_exists(MODEL_DIR / "grader_minilm"))
print("Cached router model:", _model_artifact_exists(MODEL_DIR / "router_distilbert"))

Fine-tuning DATA_DIR: /content/drive/MyDrive/AI574/finetune_artifacts/data/finetune
Fine-tuning MODEL_DIR: /content/drive/MyDrive/AI574/finetune_artifacts/models
Cached grader data: True
Cached router data: True
Cached grader model: True
Cached router model: True


In [32]:
# Optional training/reuse cell merged from Finetune_Grader_and_Router.ipynb.
# Defaults to skip training in the clean submission notebook.
if not RUN_LEARNED_MODEL_TRAINING:
    print("Skipping learned grader/router training. Set RUN_LEARNED_MODEL_TRAINING=True to train or refresh artifacts.")
else:
    from sentence_transformers import CrossEncoder
    from sentence_transformers.readers import InputExample
    from torch.utils.data import DataLoader
    from datasets import Dataset
    from transformers import (
        AutoModelForSequenceClassification,
        AutoTokenizer,
        DataCollatorWithPadding,
        Trainer,
        TrainingArguments,
    )
    import inspect

    # --- Train or reuse learned document grader ---
    required_grader_files = ["grader_train.jsonl", "grader_val.jsonl"]
    if not all(_data_artifact_exists(name) for name in required_grader_files):
        raise FileNotFoundError(
            "Missing grader_train.jsonl or grader_val.jsonl. Run data synthesis in "
            "Finetune_Grader_and_Router.ipynb first."
        )

    grader_train = _load_jsonl(DATA_DIR / "grader_train.jsonl")
    grade_train_pairs = [(r["query"], r["document"]) for r in grader_train]
    grade_train_labels = [int(r["label"]) for r in grader_train]
    grader_model_path = MODEL_DIR / "grader_minilm"

    if _model_artifact_exists(grader_model_path) and not FORCE_RETRAIN_GRADER:
        print("Reusing existing learned grader at", grader_model_path)
    else:
        print("Training learned grader. Output path:", grader_model_path)
        grader_ce = CrossEncoder(
            "cross-encoder/ms-marco-MiniLM-L-6-v2",
            num_labels=1,
            max_length=256,
        )
        train_samples = [
            InputExample(texts=[q, d], label=float(y))
            for (q, d), y in zip(grade_train_pairs, grade_train_labels)
        ]
        train_loader = DataLoader(train_samples, shuffle=True, batch_size=32)
        grader_ce.fit(
            train_dataloader=train_loader,
            epochs=3,
            warmup_steps=max(10, len(train_loader) // 10),
            output_path=str(grader_model_path),
        )
        grader_model_path.mkdir(parents=True, exist_ok=True)
        grader_ce.save(str(grader_model_path))
        print("Saved learned grader to", grader_model_path)

    # --- Train or reuse learned domain router ---
    required_router_files = ["router_train.jsonl", "router_val.jsonl"]
    if not all(_data_artifact_exists(name) for name in required_router_files):
        raise FileNotFoundError(
            "Missing router_train.jsonl or router_val.jsonl. Run data synthesis in "
            "Finetune_Grader_and_Router.ipynb first."
        )

    router_train = _load_jsonl(DATA_DIR / "router_train.jsonl")
    router_val = _load_jsonl(DATA_DIR / "router_val.jsonl")
    label2id = {label: i for i, label in enumerate(ROUTER_LABELS)}
    id2label = {i: label for label, i in label2id.items()}

    train_df = pd.DataFrame(router_train)
    val_df = pd.DataFrame(router_val)
    train_df["label_id"] = train_df["label"].map(label2id)
    val_df["label_id"] = val_df["label"].map(label2id)

    train_ds = Dataset.from_pandas(train_df[["query", "label_id"]].rename(columns={"query": "text", "label_id": "label"}))
    val_ds = Dataset.from_pandas(val_df[["query", "label_id"]].rename(columns={"query": "text", "label_id": "label"}))

    tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

    def tokenize_batch(examples):
        return tokenizer(examples["text"], truncation=True, max_length=128)

    train_ds = train_ds.map(tokenize_batch, batched=True)
    val_ds = val_ds.map(tokenize_batch, batched=True)

    router_model_path = MODEL_DIR / "router_distilbert"
    if _model_artifact_exists(router_model_path) and not FORCE_RETRAIN_ROUTER:
        print("Reusing existing learned router at", router_model_path)
    else:
        print("Training learned router. Output path:", router_model_path)
        router_model = AutoModelForSequenceClassification.from_pretrained(
            "distilbert-base-uncased",
            num_labels=len(ROUTER_LABELS),
            id2label=id2label,
            label2id=label2id,
        )

        def compute_router_metrics(eval_pred):
            logits, labels = eval_pred
            preds = np.argmax(logits, axis=-1)
            acc = accuracy_score(labels, preds)
            p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="macro", zero_division=0)
            return {"accuracy": acc, "precision_macro": p, "recall_macro": r, "f1_macro": f1}

        base_args = dict(
            output_dir=str(router_model_path / "checkpoints"),
            learning_rate=2e-5,
            per_device_train_batch_size=16,
            per_device_eval_batch_size=16,
            num_train_epochs=3,
            weight_decay=0.01,
            logging_steps=50,
            load_best_model_at_end=True,
            metric_for_best_model="f1_macro",
            report_to="none",
        )
        param_names = set(inspect.signature(TrainingArguments.__init__).parameters.keys())
        if "evaluation_strategy" in param_names:
            base_args["evaluation_strategy"] = "epoch"
        elif "eval_strategy" in param_names:
            base_args["eval_strategy"] = "epoch"
        else:
            base_args["do_eval"] = True
        if "save_strategy" in param_names:
            base_args["save_strategy"] = "epoch"
        args = TrainingArguments(**{k: v for k, v in base_args.items() if k in param_names})

        trainer = Trainer(
            model=router_model,
            args=args,
            train_dataset=train_ds,
            eval_dataset=val_ds,
            tokenizer=tokenizer,
            data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
            compute_metrics=compute_router_metrics,
        )
        trainer.train()
        trainer.save_model(str(router_model_path))
        tokenizer.save_pretrained(str(router_model_path))
        print("Saved learned router to", router_model_path)

Skipping learned grader/router training. Set RUN_LEARNED_MODEL_TRAINING=True to train or refresh artifacts.


In [33]:
# Optional held-out evaluation merged from Finetune_Grader_and_Router.ipynb.
# Uses cached models/data and compares learned components to LLM-based components.
if not RUN_LEARNED_MODEL_EVALUATION:
    print("Skipping learned grader/router evaluation. Set RUN_LEARNED_MODEL_EVALUATION=True to rerun held-out comparisons.")
else:
    import torch
    import seaborn as sns
    from sentence_transformers import CrossEncoder
    from transformers import AutoModelForSequenceClassification, AutoTokenizer
    from langchain_core.documents import Document
    from rag_core.document_grader import DocumentGrader
    from orchestration.supervisor import SupervisorAgent

    # --- Learned grader vs LLM DocumentGrader latency/quality ---
    if not _data_artifact_exists("grader_gold.jsonl"):
        raise FileNotFoundError("Missing grader_gold.jsonl. Run the fine-tuning data synthesis first.")
    grader_gold = _load_jsonl(DATA_DIR / "grader_gold.jsonl")
    grader_model_dir = _resolve_saved_model_dir("grader_minilm")
    learned_grader = CrossEncoder(str(grader_model_dir))
    llm_grader = DocumentGrader(llm=llm)

    def learned_predict_label(query: str, doc: str, threshold: float = 0.5) -> int:
        score = float(learned_grader.predict([(query, doc)])[0])
        return int(score >= threshold)

    y_true_grader = [int(r["label"]) for r in grader_gold]
    y_pred_grader = [learned_predict_label(r["query"], r["document"]) for r in grader_gold]
    grader_acc = accuracy_score(y_true_grader, y_pred_grader)
    grader_p, grader_r, grader_f1, _ = precision_recall_fscore_support(
        y_true_grader,
        y_pred_grader,
        average="binary",
        zero_division=0,
    )

    latency_subset = grader_gold[:min(200, len(grader_gold))]
    t0 = time.perf_counter()
    _ = [learned_predict_label(x["query"], x["document"]) for x in latency_subset]
    learned_grader_ms = (time.perf_counter() - t0) * 1000 / max(1, len(latency_subset))

    t0 = time.perf_counter()
    for x in latency_subset:
        _ = llm_grader.grade_single(x["query"], Document(page_content=x["document"], metadata={}))
    llm_grader_ms = (time.perf_counter() - t0) * 1000 / max(1, len(latency_subset))

    print("Learned grader metrics:", {
        "accuracy": grader_acc,
        "precision": grader_p,
        "recall": grader_r,
        "f1": grader_f1,
        "learned_ms_per_pair": learned_grader_ms,
        "llm_ms_per_pair": llm_grader_ms,
    })

    cm = confusion_matrix(y_true_grader, y_pred_grader, labels=[0, 1])
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["irrelevant", "relevant"], yticklabels=["irrelevant", "relevant"])
    plt.title("Learned Grader Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.show()

    # --- Learned DistilBERT router vs LLM-only SupervisorAgent ---
    if not _data_artifact_exists("router_gold.jsonl"):
        raise FileNotFoundError("Missing router_gold.jsonl. Run the fine-tuning data synthesis first.")
    router_gold = _load_jsonl(DATA_DIR / "router_gold.jsonl")
    router_model_dir = _resolve_saved_model_dir("router_distilbert")
    label2id = {label: i for i, label in enumerate(ROUTER_LABELS)}
    id2label = {i: label for label, i in label2id.items()}
    router_tokenizer = AutoTokenizer.from_pretrained(str(router_model_dir))
    router_model = AutoModelForSequenceClassification.from_pretrained(str(router_model_dir))
    router_model.eval()
    supervisor = SupervisorAgent(llm=llm)

    def predict_router_distilbert(query: str) -> str:
        inputs = router_tokenizer(query, return_tensors="pt", truncation=True, max_length=128)
        with torch.no_grad():
            logits = router_model(**inputs).logits
        pred_id = int(torch.argmax(logits, dim=-1).item())
        return id2label[pred_id]

    def predict_router_llm_only(query: str) -> str:
        out = supervisor.route(query)
        domain = out.get("domain", "fallback")
        return domain if domain in ROUTER_LABELS else "fallback"

    def evaluate_router(name: str, predict_fn) -> dict:
        t0 = time.perf_counter()
        preds = [predict_fn(row["query"]) for row in router_gold]
        ms_per_query = (time.perf_counter() - t0) * 1000 / max(1, len(router_gold))
        y_true = [row["label"] for row in router_gold]
        acc = accuracy_score(y_true, preds)
        p, r, f1, _ = precision_recall_fscore_support(y_true, preds, average="macro", zero_division=0)
        return {
            "router": name,
            "accuracy": acc,
            "precision_macro": p,
            "recall_macro": r,
            "f1_macro": f1,
            "ms_per_query": ms_per_query,
            "preds": preds,
        }

    router_comparison = [
        evaluate_router("DistilBERT fine-tuned router", predict_router_distilbert),
        evaluate_router("LLM-only supervisor router", predict_router_llm_only),
    ]
    router_metrics_df = pd.DataFrame([
        {k: v for k, v in row.items() if k != "preds"}
        for row in router_comparison
    ])
    display(router_metrics_df)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    y_true_router = [row["label"] for row in router_gold]
    for ax, row in zip(axes, router_comparison):
        cm = confusion_matrix(y_true_router, row["preds"], labels=ROUTER_LABELS)
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=ROUTER_LABELS, yticklabels=ROUTER_LABELS, ax=ax)
        ax.set_title(row["router"])
        ax.set_xlabel("Predicted")
        ax.set_ylabel("True")
    plt.tight_layout()
    plt.show()

Skipping learned grader/router evaluation. Set RUN_LEARNED_MODEL_EVALUATION=True to rerun held-out comparisons.


In [34]:
# Optional runtime adapters merged from Finetune_Grader_and_Router.ipynb.
# These preserve the existing CRAG interfaces and are disabled by default.
if not (USE_LEARNED_GRADER or USE_LEARNED_ROUTER):
    print("Learned runtime adapters are disabled. Set USE_LEARNED_GRADER=True and/or USE_LEARNED_ROUTER=True for a smoke test.")
else:
    import torch
    from sentence_transformers import CrossEncoder
    from transformers import AutoModelForSequenceClassification, AutoTokenizer
    from langchain_core.documents import Document
    from orchestration.hybrid_router import HybridRoutePrediction
    from orchestration import workflow_graph as wg
    from rag_core import crag_pipeline as cp
    from ingestion.index_builder import IndexBuilder

    class LearnedDocumentGrader:
        def __init__(self, model_path: Path, threshold: float = 0.5):
            self.model = CrossEncoder(str(model_path))
            self.threshold = threshold

        def grade_documents(self, query: str, documents: List[Document]) -> Dict[str, List[Document]]:
            if not documents:
                return {"relevant": [], "irrelevant": [], "ambiguous": [], "grades": []}
            pairs = [(query, doc.page_content[:2000]) for doc in documents]
            scores = self.model.predict(pairs)
            buckets = {"relevant": [], "irrelevant": [], "ambiguous": []}
            grades = []
            for i, (doc, score_value) in enumerate(zip(documents, scores)):
                score = float(score_value)
                relevance = "relevant" if score >= self.threshold else "irrelevant"
                buckets[relevance].append(doc)
                grades.append({"doc_index": i, "grade": {"relevance": relevance, "score": score, "reasoning": "cross-encoder"}})
            return {**buckets, "grades": grades}

    class LearnedDomainClassifier:
        def __init__(self, model_path: Path, label2id: Dict[str, int], min_confidence: float = 0.70, min_margin: float = 0.15):
            self.label2id = label2id
            self.id2label = {v: k for k, v in label2id.items()}
            self.min_confidence = min_confidence
            self.min_margin = min_margin
            self.tokenizer = AutoTokenizer.from_pretrained(str(model_path))
            self.model = AutoModelForSequenceClassification.from_pretrained(str(model_path))
            self.model.eval()

        def predict(self, query: str) -> HybridRoutePrediction:
            inputs = self.tokenizer(query, return_tensors="pt", truncation=True, max_length=128)
            with torch.no_grad():
                logits = self.model(**inputs).logits
                probs = torch.softmax(logits, dim=-1).cpu().numpy()[0]
            top2 = np.argsort(-probs)[:2]
            top_idx, second_idx = int(top2[0]), int(top2[1])
            top_domain = self.id2label[top_idx]
            second_domain = self.id2label[second_idx]
            top_conf = float(probs[top_idx])
            second_conf = float(probs[second_idx])
            margin = top_conf - second_conf
            accepted = (
                top_domain in {"industrial", "recipe", "scientific"}
                and top_conf >= self.min_confidence
                and margin >= self.min_margin
            )
            return HybridRoutePrediction(
                domain=top_domain if top_domain in {"industrial", "recipe", "scientific"} else "fallback",
                confidence=top_conf,
                second_domain=second_domain,
                second_confidence=second_conf,
                margin=margin,
                scores={self.id2label[i]: float(probs[i]) for i in range(len(probs))},
                accepted=accepted,
                reason=f"Learned classifier predicted {top_domain} (confidence={top_conf:.2f}, margin={margin:.2f}).",
            )

    runtime_grader = LearnedDocumentGrader(_resolve_saved_model_dir("grader_minilm")) if USE_LEARNED_GRADER else None
    runtime_router = LearnedDomainClassifier(
        _resolve_saved_model_dir("router_distilbert"),
        label2id={label: i for i, label in enumerate(ROUTER_LABELS)},
    ) if USE_LEARNED_ROUTER else None

    orig_supervisor_cls = wg.SupervisorAgent
    orig_doc_grader_cls = cp.DocumentGrader

    try:
        if USE_LEARNED_ROUTER:
            def _supervisor_factory(llm_obj):
                return orig_supervisor_cls(llm_obj, hybrid_router=runtime_router, enable_hybrid_router=True)
            wg.SupervisorAgent = _supervisor_factory

        if USE_LEARNED_GRADER:
            class _NotebookDocumentGrader:
                def __init__(self, llm_obj):
                    self._delegate = runtime_grader

                def grade_documents(self, query, documents):
                    return self._delegate.grade_documents(query, documents)

                def grade_single(self, query, document):
                    one = self.grade_documents(query, [document])
                    if one["relevant"]:
                        return {"relevance": "relevant", "score": 1.0, "reasoning": "learned"}
                    return {"relevance": "irrelevant", "score": 0.0, "reasoning": "learned"}

                def has_sufficient_context(self, grading_result):
                    return len(grading_result.get("relevant", [])) >= 1

            cp.DocumentGrader = _NotebookDocumentGrader

        learned_wf = wg.build_workflow(llm, vs, builder)
        sample_query = "PowerFlex 525 F004 fault during startup"
        learned_result = wg.run_query(learned_wf, sample_query)

        print("USE_LEARNED_GRADER =", USE_LEARNED_GRADER)
        print("USE_LEARNED_ROUTER =", USE_LEARNED_ROUTER)
        print("Sample query:", sample_query)
        display_result(learned_result)
    finally:
        wg.SupervisorAgent = orig_supervisor_cls
        cp.DocumentGrader = orig_doc_grader_cls

Learned runtime adapters are disabled. Set USE_LEARNED_GRADER=True and/or USE_LEARNED_ROUTER=True for a smoke test.


In [ ]:
# LLM-based vs learned grader/router comparison (§21.5).
# Same completed-run metrics as `Finetune_Grader_and_Router.ipynb`; no retraining required.
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

learned_extension_results = pd.DataFrame(
    [
        {
            "decision_point": "document_relevance_grading",
            "llm_component": "DocumentGrader (LLM-as-judge)",
            "learned_component": "MiniLM cross-encoder",
            "artifact_subdir": "grader_minilm",
            "learned_accuracy": 0.899,
            "learned_f1": 0.882,
            "llm_accuracy": None,
            "llm_f1": None,
            "quality_note": "Learned grader evaluated against held-out grader labels; LLM grader is the baseline judge/label source.",
            "llm_latency_ms": 1482.81,
            "learned_latency_ms": 7.21,
        },
        {
            "decision_point": "domain_routing",
            "llm_component": "SupervisorAgent.route (LLM)",
            "learned_component": "DistilBERT 5-class router",
            "artifact_subdir": "router_distilbert",
            "learned_accuracy": 1.000,
            "learned_f1": 1.000,
            "llm_accuracy": 0.845,
            "llm_f1": 0.795,
            "quality_note": "Both routers evaluated against held-out router labels.",
            "llm_latency_ms": 1088.57,
            "learned_latency_ms": 19.39,
        },
    ]
)

learned_extension_results["speedup_x"] = (
    learned_extension_results["llm_latency_ms"] /
    learned_extension_results["learned_latency_ms"]
)

comparison_display = learned_extension_results[
    [
        "decision_point",
        "llm_component",
        "learned_component",
        "artifact_subdir",
        "llm_accuracy",
        "learned_accuracy",
        "llm_f1",
        "learned_f1",
        "llm_latency_ms",
        "learned_latency_ms",
        "speedup_x",
        "quality_note",
    ]
]

display(
    comparison_display.style.format(
        {
            "llm_accuracy": "{:.3f}",
            "learned_accuracy": "{:.3f}",
            "llm_f1": "{:.3f}",
            "learned_f1": "{:.3f}",
            "llm_latency_ms": "{:.2f}",
            "learned_latency_ms": "{:.2f}",
            "speedup_x": "{:.1f}x",
        },
        na_rep="n/a",
    ).hide(axis="index")
)

# Latency is the clearest operational gain: lower is better.
latency_plot = comparison_display.set_index("decision_point")[["llm_latency_ms", "learned_latency_ms"]]
ax = latency_plot.plot(kind="bar", figsize=(8, 4), rot=15)
ax.set_title("LLM-Based vs Learned Components: Latency")
ax.set_ylabel("Milliseconds")
ax.legend(["LLM-based", "Learned"])
plt.tight_layout()
plt.show()

# Router quality has both LLM and learned scores; grader quality is shown for the learned model only.
quality_plot = comparison_display.set_index("decision_point")[["llm_f1", "learned_f1"]]
ax = quality_plot.plot(kind="bar", figsize=(8, 4), rot=15, ylim=(0, 1.05))
ax.set_title("LLM-Based vs Learned Components: F1")
ax.set_ylabel("F1 score")
ax.legend(["LLM-based", "Learned"])
plt.tight_layout()
plt.show()


<a id="216-five-query-end-to-end-comparison"></a>

## 21.6 Five-Query Demo Summary

The five demo queries cover the required behaviors:

| # | Behavior | Query type |
|---|---|---|
| 1 | Industrial | PowerFlex fault troubleshooting |
| 2 | Recipe | Egg substitute in pancakes |
| 3 | Scientific | RAG research summary |
| 4 | Synthesis | Scientific summary plus dinner recipe |
| 5 | Fallback | Current-events question outside local corpora |

Takeaway: routing chooses the right path, CRAG adds grounding checks, synthesis handles multi-domain requests, and fallback avoids forcing out-of-scope questions into local data.

<a id="217-formal-150-query-evaluation"></a>

## 21.7 Formal 150-Query Evaluation

Benchmark setup:

- 150 queries: 50 industrial, 50 recipe, 50 scientific.
- Systems: `multi_agent_crag` vs `monolithic_rag`.
- Metrics: route accuracy, retrieval P@5, raw hallucination, unblocked hallucination, task completion, and latency.

`RUN_FORMAL_EVAL` is `False` by default; use the cached final results below unless intentionally rerunning the full benchmark.


In [ ]:
# Purpose: define and optionally run the 150-query formal evaluation benchmark.
# This benchmark compares the multi-agent CRAG workflow against a single
# monolithic RAG baseline on routing accuracy, retrieval precision,
# hallucination rate, task completion, and latency.

import json
import os
import re
import sys
import time
from pathlib import Path
from collections import Counter, defaultdict
from statistics import mean
from langchain_core.messages import HumanMessage


def ensure_project_import_path() -> None:
    """Make project packages importable when this cell is run out of order."""
    candidates = [
        globals().get("GIT_REPO"),
        "/content/AI574",
        os.getcwd(),
        str(Path.cwd().parent),
    ]
    for candidate in candidates:
        if candidate and os.path.isdir(os.path.join(candidate, "config")):
            if candidate not in sys.path:
                sys.path.insert(0, candidate)
            return
    raise RuntimeError(
        "Could not find the AI574 project root. Run the clone/update and project import-path cells first."
    )


ensure_project_import_path()

EVAL_DOMAINS = ["industrial", "recipe", "scientific"]

INDUSTRIAL_EVAL_QUERIES = [
    "My PowerFlex 525 drive shows fault F004. What should I check?",
    "How do I troubleshoot a PowerFlex 525 undervoltage fault?",
    "What causes fault F005 overvoltage on an Allen-Bradley drive?",
    "PanelView Plus says Controller not found. What should I inspect?",
    "How do I verify EtherNet/IP communication to a CompactLogix controller?",
    "What should I check when a VFD trips on overcurrent during acceleration?",
    "How do I safely inspect input power terminals on a PowerFlex drive?",
    "What does a flashing ENET light mean on a CompactLogix controller?",
    "How can I back up PowerFlex 755 drive parameters before maintenance?",
    "What preventive maintenance should I perform on a PowerFlex 755 drive?",
    "How do I troubleshoot a motor drive that faults after deceleration?",
    "What checks are recommended for loose VFD input power connections?",
    "How do I review a PowerFlex drive fault history from parameters?",
    "What should I do if a PanelView shows stale PLC data?",
    "How do I check whether a controller is in Run mode for an HMI?",
    "What can cause an auxiliary input fault on a PowerFlex 525?",
    "How do I reset an Allen-Bradley drive after clearing a fault?",
    "What safety steps are needed before servicing a VFD cabinet?",
    "How do I diagnose an EtherNet/IP device that does not appear in RSLinx?",
    "What should I verify when a managed switch VLAN isolates PLC devices?",
    "How do I measure line voltage balance for a three-phase drive?",
    "What does DC bus voltage indicate on a PowerFlex drive?",
    "How do I troubleshoot a drive that trips when the motor regenerates energy?",
    "When should I add a dynamic brake resistor to a VFD system?",
    "How do I inspect cooling fans on a PowerFlex drive?",
    "What are signs that drive capacitors may be degraded?",
    "How do I configure an IP address for a CompactLogix controller?",
    "What should I check when an HMI and PLC are on different subnets?",
    "How do I confirm the shortcut path in FactoryTalk View Studio?",
    "What should I inspect if an Ethernet cable or switch port has no link light?",
    "How do I troubleshoot a controller-to-controller produced consumed tag issue?",
    "What steps should I follow after replacing a drive cooling fan?",
    "How do I verify firmware compatibility for Rockwell Automation equipment?",
    "What critical spares should I stock for a PowerFlex drive system?",
    "How do I troubleshoot an external device wired to an auxiliary input?",
    "What parameter controls deceleration time on a PowerFlex drive?",
    "How do I enable bus regulation for overvoltage faults?",
    "What should I check when a drive input fuse is blown?",
    "How do I validate a PLC network path before downloading a program?",
    "What should I do before tightening drive terminal connections?",
    "How do I identify whether an HMI communication failure is physical or configuration related?",
    "What is the safe wait time before touching DC bus components?",
    "How do I troubleshoot motor overload trips in a Rockwell drive system?",
    "What should I check if a drive fault repeats after reset?",
    "How do I confirm a PowerFlex drive is within nameplate input voltage?",
    "What information should be recorded from a VFD fault log?",
    "How do I handle a PLC communication problem after changing IP addresses?",
    "What is the difference between Program mode and Run mode for live HMI data?",
    "How do I troubleshoot an industrial Ethernet device on a wrong subnet?",
    "What routine checks should be included in a drive cabinet inspection?",
]

RECIPE_EVAL_QUERIES = [
    "What can I substitute for eggs in pancakes?",
    "Give me a quick vegetarian pasta recipe for a weeknight dinner.",
    "How long should I bake chicken thighs at 400F?",
    "Suggest a low-sodium lentil soup with carrots.",
    "What is a gluten-free substitute for regular flour in pizza dough?",
    "How can I make pancakes without buttermilk?",
    "Give me a high-protein breakfast recipe with eggs.",
    "What can I use instead of butter in cookies?",
    "Suggest a dairy-free dessert recipe with chocolate.",
    "How do I make a quick chicken stir-fry?",
    "What is a good vegetarian one-pot dinner?",
    "How can I reduce sugar in a cookie recipe?",
    "Suggest a keto-friendly breakfast recipe.",
    "What are good substitutes for milk in pancakes?",
    "How do I make homemade Margherita pizza?",
    "Give me a quick soup recipe using pantry ingredients.",
    "What can replace bananas in banana bread?",
    "How can I make a recipe gluten-free?",
    "Suggest a low-carb dinner using chicken.",
    "How do I cook rice for a stir-fry meal?",
    "What is an easy dessert for someone avoiding eggs?",
    "Give me a vegetarian recipe with mushrooms and garlic.",
    "How can I make pancakes fluffier?",
    "What is a healthy snack using oats?",
    "Suggest a quick lunch with chickpeas.",
    "How do I adapt a recipe for a dairy allergy?",
    "What can I cook with bell peppers and chicken?",
    "Give me a simple tomato sauce recipe.",
    "How do I make soup thicker without cream?",
    "What is a good replacement for soy sauce?",
    "Suggest a recipe for gluten-free cookies.",
    "How do I make a quick vegetable saute?",
    "What breakfast can I make with applesauce?",
    "Give me a low-calorie dinner idea with vegetables.",
    "How can I make pizza dough rise properly?",
    "What can I substitute for yeast in bread?",
    "Suggest a fast dinner using pasta and garlic.",
    "How do I make pancakes for one person?",
    "What can replace sour cream in a recipe?",
    "Give me a recipe for chicken with rice and vegetables.",
    "How do I make a vegan pancake batter?",
    "What ingredients make cookies chewy?",
    "Suggest a recipe using lentils for meal prep.",
    "How can I reduce sodium in soup?",
    "What is a quick vegetarian side dish?",
    "Give me a breakfast recipe that is high in protein and low in carbs.",
    "How do I replace wheat flour for a gluten-free guest?",
    "What can I make with carrots, beans, and rice?",
    "Suggest a simple dinner that takes less than 30 minutes.",
    "How do I choose an egg substitute for baking?",
]

SCIENTIFIC_EVAL_QUERIES = [
    "Summarize recent research on retrieval-augmented generation for NLP.",
    "What are the main evaluation challenges for RAG systems?",
    "Compare LoRA and full fine-tuning for instruction-following models.",
    "Summarize recent transformer papers on ArXiv.",
    "What does recent research say about hallucination detection in LLMs?",
    "Find papers about retrieval-augmented question answering.",
    "What are common metrics for evaluating generated text factuality?",
    "Summarize work on dense retrieval for open-domain QA.",
    "What are recent methods for improving robustness in RAG?",
    "Compare supervised fine-tuning and reinforcement learning from human feedback.",
    "What is the role of reranking in neural information retrieval?",
    "Summarize recent research on cross-encoders for relevance grading.",
    "What papers discuss query rewriting for retrieval systems?",
    "Explain recent work on long-context language models.",
    "What does recent NLP research say about citation support in generated answers?",
    "Find scientific papers about domain-specific question answering.",
    "Summarize research on graph neural networks for recommendation.",
    "What are recent approaches to parameter-efficient fine-tuning?",
    "Compare embedding models for semantic search.",
    "What are common datasets for retrieval-augmented generation evaluation?",
    "Summarize recent research on document reranking.",
    "What papers study hallucinations in summarization systems?",
    "Explain recent work on instruction tuning for language models.",
    "What are recent findings on prompt engineering for LLMs?",
    "Summarize scientific work on neural machine translation with retrieval.",
    "What papers discuss uncertainty estimation in language models?",
    "Find recent research on LLM evaluation benchmarks.",
    "Summarize work on multi-agent language model systems.",
    "What are recent methods for reducing hallucinations in RAG?",
    "Compare sparse retrieval and dense retrieval in NLP systems.",
    "What are recent papers on scientific literature search?",
    "Summarize research on contrastive learning for sentence embeddings.",
    "What papers discuss question decomposition for RAG?",
    "Explain current research on retrieval fusion methods.",
    "What are recent trends in ArXiv papers about transformers?",
    "Summarize research about evaluating source attribution in generated answers.",
    "What does recent work say about tool-using language agents?",
    "Find papers about self-correction in language models.",
    "Summarize research on retrieval-augmented dialogue systems.",
    "What are common failure modes in retrieval-augmented generation?",
    "Compare generative retrieval and traditional retrieval pipelines.",
    "What papers discuss factual consistency in abstractive summarization?",
    "Summarize recent work on efficient LLM inference.",
    "What are recent methods for training smaller task-specific classifiers from LLM labels?",
    "Find papers about knowledge-intensive NLP tasks.",
    "What research studies domain adaptation for language models?",
    "Summarize recent work on semantic similarity models.",
    "What papers compare RAG against closed-book language models?",
    "Explain recent research on grounded response generation.",
    "What are best practices for evaluating retrieval precision in RAG?",
]

assert len(INDUSTRIAL_EVAL_QUERIES) == 50
assert len(RECIPE_EVAL_QUERIES) == 50
assert len(SCIENTIFIC_EVAL_QUERIES) == 50

FORMAL_EVAL_QUERIES = [
    {"id": f"industrial_{i:02d}", "domain": "industrial", "query": q}
    for i, q in enumerate(INDUSTRIAL_EVAL_QUERIES, 1)
] + [
    {"id": f"recipe_{i:02d}", "domain": "recipe", "query": q}
    for i, q in enumerate(RECIPE_EVAL_QUERIES, 1)
] + [
    {"id": f"scientific_{i:02d}", "domain": "scientific", "query": q}
    for i, q in enumerate(SCIENTIFIC_EVAL_QUERIES, 1)
]

assert len(FORMAL_EVAL_QUERIES) == 150
print(f"Formal evaluation set ready: {len(FORMAL_EVAL_QUERIES)} queries")
print(Counter(case["domain"] for case in FORMAL_EVAL_QUERIES))

DOMAIN_RELEVANCE_TERMS = {
    "industrial": {"drive", "fault", "powerflex", "controller", "plc", "vfd", "ethernet", "panelview", "voltage", "motor", "terminal", "safety"},
    "recipe": {"recipe", "cook", "bake", "ingredient", "substitute", "pancake", "chicken", "vegetarian", "gluten", "dinner", "breakfast", "soup"},
    "scientific": {"paper", "research", "rag", "retrieval", "generation", "transformer", "llm", "evaluation", "arxiv", "model", "nlp", "fine-tuning"},
}

STOPWORDS = {
    "what", "when", "where", "which", "with", "from", "that", "this", "should", "could",
    "would", "about", "into", "using", "recent", "give", "make", "does", "good", "best",
}


def _query_terms(query, domain):
    """Create a lightweight relevance rubric from the query and domain vocabulary."""
    tokens = [t for t in re.findall(r"[a-zA-Z][a-zA-Z0-9+-]{3,}", query.lower()) if t not in STOPWORDS]
    return set(tokens[:8]) | DOMAIN_RELEVANCE_TERMS[domain]


def _doc_text(doc):
    """Flatten a retrieved document and metadata into searchable lowercase text."""
    metadata = getattr(doc, "metadata", {}) or {}
    meta_text = " ".join(str(v) for v in metadata.values() if v is not None)
    return f"{getattr(doc, 'page_content', '')} {meta_text}".lower()


def _is_relevant_doc(doc, expected_domain, query):
    """Proxy relevance: expected domain plus at least one query/domain term match."""
    if getattr(doc, "metadata", {}).get("eval_domain") != expected_domain:
        return False
    text = _doc_text(doc)
    terms = _query_terms(query, expected_domain)
    return any(term.lower() in text for term in terms)


def monolithic_retrieve(query, k=5):
    """Search every domain collection and merge hits into one monolithic top-k list."""
    hits = []
    for domain in EVAL_DOMAINS:
        for doc in vs.search(domain, query, k=k):
            doc.metadata = dict(doc.metadata or {})
            doc.metadata["eval_domain"] = domain
            score = float(doc.metadata.get("similarity_score", 0.0) or 0.0)
            hits.append((score, domain, doc))
    hits.sort(key=lambda item: item[0], reverse=True)
    return [doc for _, _, doc in hits[:k]]


def retrieval_precision_at_k(docs, expected_domain, query, k=5):
    """Fraction of top-k retrieved documents judged relevant by the proxy rubric."""
    if not docs:
        return 0.0
    top_docs = docs[:k]
    relevant = sum(_is_relevant_doc(doc, expected_domain, query) for doc in top_docs)
    return relevant / max(len(top_docs), 1)


def run_monolithic_rag(query, k=5, max_context_chars=10_000):
    """Single monolithic RAG baseline: no router, no grading, no rewrite, no validation."""
    t0 = time.perf_counter()
    docs = monolithic_retrieve(query, k=k)
    retrieve_s = time.perf_counter() - t0

    context_parts = []
    per_doc_budget = max(500, max_context_chars // max(len(docs), 1))
    for i, doc in enumerate(docs, 1):
        m = doc.metadata or {}
        label = m.get("title") or m.get("name") or m.get("source") or "unknown"
        context_parts.append(
            f"[Document {i} | Domain: {m.get('eval_domain')} | Source: {label}]\n"
            f"{doc.page_content[:per_doc_budget]}"
        )
    context = "\n\n---\n\n".join(context_parts)

    prompt = f"""Answer the user query using only the retrieved context.
If the context is insufficient, say so clearly. Cite document numbers when possible.

Retrieved Context:
{context}

User Query: {query}
"""
    generate_t0 = time.perf_counter()
    response = llm.invoke([HumanMessage(content=prompt)]).content.strip()
    generate_s = time.perf_counter() - generate_t0

    return {
        "domain": docs[0].metadata.get("eval_domain") if docs else "none",
        "response": response,
        "docs": docs,
        "timing": {
            "retrieve_s": retrieve_s,
            "generate_s": generate_s,
            "total_s": time.perf_counter() - t0,
        },
    }


def _proxy_quality_judge(response, docs_or_sources):
    """Cheap fallback quality judge used when the LLM evaluator is disabled."""
    text = (response or "").lower()
    has_answer = len(text.split()) >= 25
    admits_failure = any(phrase in text for phrase in ["insufficient", "could not", "cannot answer", "not enough context"])
    has_sources = bool(docs_or_sources)
    return {
        "hallucinated": not has_sources,
        "task_complete": has_answer and not admits_failure,
        "judge_reason": "proxy: answer length/source/failure-phrase heuristic",
    }


def _format_sources_for_judge(result_sources=None, docs=None):
    """Combine exact result source metadata with retrieved text for the judge."""
    parts = []
    for i, src in enumerate(result_sources or [], 1):
        if isinstance(src, dict):
            label = src.get("title") or src.get("name") or src.get("source") or src.get("url") or "unknown"
            parts.append(f"[Result Source {i}] {label} | metadata={src}")
        else:
            parts.append(f"[Result Source {i}] {src}")
    for i, doc in enumerate(docs or [], 1):
        metadata = getattr(doc, "metadata", {}) or {}
        label = metadata.get("title") or metadata.get("name") or metadata.get("source") or "unknown"
        parts.append(
            f"[Retrieved Doc {i}] {label} | metadata={metadata}\n"
            f"{getattr(doc, 'page_content', '')[:1500]}"
        )
    return "\n\n".join(parts)


def _task_completion_heuristic(query, response, expected_domain=None):
    """Domain-aware fallback for obvious task completion cases.

    This protects the metric from an overly strict LLM judge. It does not affect
    hallucination scoring; it only asks whether the answer is useful and on task.
    """
    text = (response or "").lower()
    words = re.findall(r"[a-zA-Z][a-zA-Z0-9+-]{2,}", text)
    query_terms = _query_terms(query, expected_domain) if expected_domain in EVAL_DOMAINS else set()
    overlap = sum(1 for term in query_terms if term.lower() in text)
    refusal_terms = [
        "cannot answer", "can't answer", "i do not know", "i don't know",
        "not enough context", "insufficient context", "unable to answer",
        "outside my specialist domains",
    ]
    if len(words) < 25 or any(term in text for term in refusal_terms):
        return {
            "task_complete": False,
            "reason": "heuristic: too short or explicit refusal/insufficient-context answer",
        }

    domain_markers = {
        "industrial": ["check", "inspect", "measure", "verify", "troubleshoot", "fault", "safety", "voltage", "terminal", "controller", "drive"],
        "recipe": ["recipe", "ingredient", "substitute", "cook", "bake", "steps", "minutes", "cup", "use", "serve"],
        "scientific": ["research", "paper", "study", "method", "summar", "model", "retrieval", "evaluation", "finding", "approach"],
    }
    markers = domain_markers.get(expected_domain, [])
    marker_hits = sum(1 for marker in markers if marker in text)
    complete = marker_hits >= 2 and overlap >= 1
    return {
        "task_complete": complete,
        "reason": f"heuristic: marker_hits={marker_hits}, query/domain_overlap={overlap}, words={len(words)}",
    }


def judge_with_llm(query, response, sources_text, expected_domain=None):
    """Optional LLM-as-judge for hallucination and task completion.

    The rubric intentionally separates factual grounding from task completion:
    an answer can be useful and complete while still containing an unsupported
    extra claim that should count as hallucination. A domain-aware heuristic can
    override false-negative task-completion judgments for clearly useful answers.
    """
    judge_prompt = f"""You are evaluating a RAG answer.
Return only valid JSON with keys: hallucinated, task_complete, reason.

Definitions:
- hallucinated=true if the answer contains important factual or procedural claims that are not supported by the provided sources.
- task_complete=true if the answer gives a relevant, actionable response to the user query.
- Judge task_complete from the Query and Answer first; use Sources mainly for hallucination/grounding.
- Safety warnings, caveats, and recommendations to verify procedures should NOT make task_complete=false if the answer still gives useful steps.
- An answer can be task_complete=true and hallucinated=true at the same time if it answers the task but includes unsupported extra claims.
- Mark task_complete=false only if the answer refuses, is mostly irrelevant, says it cannot answer, or omits the main requested guidance.

Domain-specific task-completion guidance:
- Industrial troubleshooting is complete when it identifies likely causes or concrete checks and includes appropriate safety framing.
- Recipe guidance is complete when it gives usable substitutions, ingredients, steps, or cooking guidance relevant to the query.
- Scientific summarization is complete when it summarizes relevant research themes, papers, methods, or evaluation criteria.

Expected domain: {expected_domain}
Query: {query}

Answer:
{response}

Sources:
{sources_text[:10000]}
"""
    heuristic = _task_completion_heuristic(query, response, expected_domain)
    try:
        raw = llm.invoke([HumanMessage(content=judge_prompt)]).content.strip()
        match = re.search(r"\{.*\}", raw, flags=re.S)
        data = json.loads(match.group(0) if match else raw)
        llm_task_complete = bool(data.get("task_complete", False))
        task_complete = llm_task_complete or heuristic["task_complete"]
        source = "llm" if llm_task_complete else ("heuristic_override" if task_complete else "llm_and_heuristic_false")
        return {
            "hallucinated": bool(data.get("hallucinated", False)),
            "task_complete": task_complete,
            "task_completion_source": source,
            "judge_reason": (
                f"llm: {str(data.get('reason', '')).strip()} | "
                f"{heuristic['reason']}"
            ).strip(),
        }
    except Exception as exc:
        print(f"Judge failed; using proxy judge instead: {exc}")
        proxy = _proxy_quality_judge(response, sources_text)
        proxy["task_complete"] = proxy["task_complete"] or heuristic["task_complete"]
        proxy["task_completion_source"] = "proxy_or_heuristic"
        proxy["judge_reason"] = f"{proxy.get('judge_reason', '')} | {heuristic['reason']}"
        return proxy


def summarize_eval_rows(rows):
    """Aggregate formal evaluation rows by system and print a report table."""
    by_system = defaultdict(list)
    for row in rows:
        by_system[row["system"]].append(row)

    summary = []
    for system, items in by_system.items():
        summary.append({
            "system": system,
            "n": len(items),
            "routing_accuracy": mean([r["routing_correct"] for r in items if r["routing_correct"] is not None]) if any(r["routing_correct"] is not None for r in items) else None,
            "retrieval_precision_at_5": mean(r["retrieval_precision_at_5"] for r in items),
            # Raw hallucination is the strict judge's unsupported-claim label.
            "raw_hallucination_rate": mean(r["hallucinated"] for r in items),
            # Reported hallucination rate is the user-facing, unblocked rate.
            # For CRAG, an unsupported answer that is detected/escalated is a
            # caught safety event, not a silent hallucination. Monolithic RAG has
            # no escalation mechanism, so raw and unblocked are the same for it.
            "hallucination_rate": mean(r.get("unblocked_hallucinated", r["hallucinated"]) for r in items),
            "task_completion_rate": mean(r["task_complete"] for r in items),
            "avg_latency_s": mean(r["latency_s"] for r in items),
        })

    print("\nFORMAL EVALUATION SUMMARY")
    print("=" * 112)
    print(f"{'System':<24} {'n':>4} {'Route Acc':>10} {'P@5':>8} {'Raw Halluc.':>12} {'Unblocked':>10} {'Task Done':>10} {'Latency':>9}")
    print("-" * 112)
    for s in summary:
        route = "n/a" if s["routing_accuracy"] is None else f"{s['routing_accuracy']:.1%}"
        print(
            f"{s['system']:<24} {s['n']:>4} {route:>10} "
            f"{s['retrieval_precision_at_5']:>8.1%} {s['raw_hallucination_rate']:>12.1%} "
            f"{s['hallucination_rate']:>10.1%} {s['task_completion_rate']:>10.1%} "
            f"{s['avg_latency_s']:>8.2f}s"
        )
    return summary


def select_balanced_eval_cases(cases, limit=None):
    """Select a near-balanced subset across domains for smoke tests.

    Example: FORMAL_EVAL_LIMIT=10 returns a 4/3/3 domain split.
    FORMAL_EVAL_LIMIT=None returns all 150 cases.
    """
    if limit is None:
        return list(cases)
    if limit <= 0:
        raise ValueError("FORMAL_EVAL_LIMIT must be positive, or None for all queries.")

    grouped = {
        domain: [case for case in cases if case["domain"] == domain]
        for domain in EVAL_DOMAINS
    }
    base_per_domain, remainder = divmod(limit, len(EVAL_DOMAINS))

    selected = []
    for i, domain in enumerate(EVAL_DOMAINS):
        take = base_per_domain + (1 if i < remainder else 0)
        selected.extend(grouped[domain][:take])

    if len(selected) < min(limit, len(cases)):
        selected_ids = {case["id"] for case in selected}
        selected.extend(case for case in cases if case["id"] not in selected_ids)
        selected = selected[:limit]

    return selected


def build_formal_eval_workflow(*, llm_supervisor_only=False):
    """Build the workflow used for formal evaluation.

    When llm_supervisor_only=True, the lexical hybrid router is disabled for this
    evaluation workflow only. The global CONFIG value is restored immediately
    after the workflow is compiled.
    """
    if not llm_supervisor_only:
        return workflow

    from config.settings import CONFIG

    previous = CONFIG.supervisor.hybrid_router_enabled
    CONFIG.supervisor.hybrid_router_enabled = False
    try:
        return build_workflow(llm=llm, vector_store=vs, index_builder=builder)
    finally:
        CONFIG.supervisor.hybrid_router_enabled = previous


def run_formal_evaluation(cases, *, workflow_to_run=None, k=5, mode="best_quality", use_llm_judge=False):
    """Run multi-agent CRAG and monolithic RAG on the same labeled query set.

    Use best_quality for formal reporting. fast_interactive caps generation at
    256 tokens, which can truncate otherwise correct answers and make task
    completion look artificially low.
    """
    rows = []
    active_workflow = workflow_to_run or workflow
    for i, case in enumerate(cases, 1):
        expected = case["domain"]
        query = case["query"]
        print(f"[{i:03d}/{len(cases)}] {case['id']} -> {expected}")

        # Multi-agent system: route through the compiled workflow.
        multi_t0 = time.perf_counter()
        multi = run_query(active_workflow, query, mode=mode, options={"k": k})
        multi_latency = time.perf_counter() - multi_t0
        routed_domain = multi.get("domain")
        routed_docs = vs.search(routed_domain, query, k=k) if routed_domain in EVAL_DOMAINS else []
        for doc in routed_docs:
            doc.metadata = dict(doc.metadata or {})
            doc.metadata["eval_domain"] = routed_domain
        multi_sources_text = _format_sources_for_judge(multi.get("sources", []), routed_docs)
        multi_quality = (
            judge_with_llm(query, multi.get("response", ""), multi_sources_text, expected_domain=expected)
            if use_llm_judge else
            _proxy_quality_judge(multi.get("response", ""), multi.get("sources", []))
        )
        multi_hallucinated = bool(multi.get("escalated", False)) if not use_llm_judge else multi_quality["hallucinated"]
        multi_escalated = bool(multi.get("escalated", False))
        rows.append({
            "system": "multi_agent_crag",
            "id": case["id"],
            "query": query,
            "expected_domain": expected,
            "predicted_domain": routed_domain,
            "routing_source": multi.get("routing_source"),
            "confidence": multi.get("confidence"),
            "routing_correct": routed_domain == expected,
            "retrieval_precision_at_5": retrieval_precision_at_k(routed_docs, expected, query, k=k),
            "hallucinated": multi_hallucinated,
            "unblocked_hallucinated": multi_hallucinated and not multi_escalated,
            "task_complete": multi_quality["task_complete"],
            "task_completion_source": multi_quality.get("task_completion_source", ""),
            "judge_reason": multi_quality.get("judge_reason", ""),
            "escalated": multi_escalated,
            "num_sources": len(multi.get("sources", []) or []),
            "response": multi.get("response", ""),
            "latency_s": (multi.get("timing") or {}).get("total_s", multi_latency),
        })

        # Monolithic baseline: search all domains as one pool and answer directly.
        mono = run_monolithic_rag(query, k=k)
        mono_sources_text = _format_sources_for_judge([], mono["docs"])
        mono_quality = judge_with_llm(query, mono["response"], mono_sources_text, expected_domain=expected) if use_llm_judge else _proxy_quality_judge(mono["response"], mono["docs"])
        rows.append({
            "system": "monolithic_rag",
            "id": case["id"],
            "query": query,
            "expected_domain": expected,
            "predicted_domain": mono.get("domain"),
            "routing_correct": None,  # monolithic baseline has no explicit router
            "retrieval_precision_at_5": retrieval_precision_at_k(mono["docs"], expected, query, k=k),
            "hallucinated": mono_quality["hallucinated"],
            "unblocked_hallucinated": mono_quality["hallucinated"],
            "task_complete": mono_quality["task_complete"],
            "task_completion_source": mono_quality.get("task_completion_source", ""),
            "judge_reason": mono_quality.get("judge_reason", ""),
            "escalated": False,
            "num_sources": len(mono.get("docs", []) or []),
            "response": mono.get("response", ""),
            "latency_s": mono["timing"]["total_s"],
        })

    summary = summarize_eval_rows(rows)
    return rows, summary


# Full final benchmark settings:
# - RUN_FORMAL_EVAL=True executes the benchmark.
# - FORMAL_EVAL_LIMIT=None uses all 150 queries.
# - USE_LLM_EVAL_JUDGE=True gives stronger hallucination/task-completion labels,
#   but adds one extra judge call per system/query and is therefore slower.
# - FORMAL_EVAL_LLM_SUPERVISOR_ONLY=True disables the hybrid lexical fast path
#   for the multi-agent system so every query is routed by the LLM supervisor.
# - FORMAL_EVAL_MODE="best_quality" avoids the 256-token truncation used by
#   fast_interactive and is the recommended setting for task-completion scoring.
RUN_FORMAL_EVAL = False  # set True to regenerate all 150×2 (slow); otherwise use cached table below
FORMAL_EVAL_LIMIT = None  # None uses all 150 queries when RUN_FORMAL_EVAL=True
USE_LLM_EVAL_JUDGE = True
FORMAL_EVAL_LLM_SUPERVISOR_ONLY = True
FORMAL_EVAL_MODE = "best_quality"

if RUN_FORMAL_EVAL:
    eval_cases = select_balanced_eval_cases(FORMAL_EVAL_QUERIES, FORMAL_EVAL_LIMIT)
    eval_workflow = build_formal_eval_workflow(
        llm_supervisor_only=FORMAL_EVAL_LLM_SUPERVISOR_ONLY
    )
    print("Evaluation domain counts:", Counter(case["domain"] for case in eval_cases))
    print("Multi-agent routing mode:", "LLM supervisor only" if FORMAL_EVAL_LLM_SUPERVISOR_ONLY else "hybrid router + LLM supervisor")
    print("Formal evaluation mode:", FORMAL_EVAL_MODE)
    formal_eval_rows, formal_eval_summary = run_formal_evaluation(
        eval_cases,
        workflow_to_run=eval_workflow,
        k=5,
        mode=FORMAL_EVAL_MODE,
        use_llm_judge=USE_LLM_EVAL_JUDGE,
    )
else:
    print("Formal benchmark is defined but not executed in normal runs.")
    print("Set RUN_FORMAL_EVAL=True and FORMAL_EVAL_LIMIT=None to run all 150 queries.")


<a id="reported-150-query-benchmark-summary-cached"></a>

## Reported 150-query Benchmark (cached)

| System | n | Route Acc | P@5 | Raw Halluc. | Unblocked | Task Done | Latency |
|--------|---|-----------|-----|-------------|-----------|-----------|--------|
| `multi_agent_crag` | 150 | 100.0% | 100.0% | 45.3% | 4.7% | 96.7% | 27.08s |
| `monolithic_rag` | 150 | n/a | 98.3% | 19.3% | 19.3% | 96.7% | 5.95s |

Takeaway: CRAG is slower, but its validation step cuts user-visible hallucination from **19.3%** to **4.7%**.


<a id="218-benchmark-visualizations"></a>

## 21.8 Benchmark Visualizations

The charts below show the same benchmark numbers visually. They use the latest run if available; otherwise they use the cached 150-query summary above.

In [ ]:
# Visualize formal evaluation results.
# Uses freshly computed `formal_eval_summary` when present; otherwise falls back
# to the cached benchmark numbers in the markdown table above.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from IPython.display import display

CACHED_FORMAL_EVAL_SUMMARY = [
    {
        "system": "multi_agent_crag",
        "n": 150,
        "routing_accuracy": 1.000,
        "retrieval_precision_at_5": 1.000,
        "raw_hallucination_rate": 0.453,
        "hallucination_rate": 0.047,
        "task_completion_rate": 0.967,
        "avg_latency_s": 27.08,
    },
    {
        "system": "monolithic_rag",
        "n": 150,
        "routing_accuracy": None,
        "retrieval_precision_at_5": 0.983,
        "raw_hallucination_rate": 0.193,
        "hallucination_rate": 0.193,
        "task_completion_rate": 0.967,
        "avg_latency_s": 5.95,
    },
]


def _summary_for_plot():
    summary = globals().get("formal_eval_summary")
    if isinstance(summary, list) and summary:
        return summary, "current formal_eval_summary"
    return CACHED_FORMAL_EVAL_SUMMARY, "cached 150-query benchmark summary"


def _pretty_system_name(system):
    names = {
        "multi_agent_crag": "Multi-agent CRAG",
        "monolithic_rag": "Monolithic RAG",
    }
    return names.get(system, str(system).replace("_", " ").title())


summary_rows, summary_source = _summary_for_plot()
summary_df = pd.DataFrame(summary_rows).copy()

numeric_cols = [
    "routing_accuracy",
    "retrieval_precision_at_5",
    "raw_hallucination_rate",
    "hallucination_rate",
    "task_completion_rate",
    "avg_latency_s",
]
for col in numeric_cols:
    summary_df[col] = pd.to_numeric(summary_df[col], errors="coerce")

print(f"Visualizing results from: {summary_source}")
display(
    summary_df.assign(system=summary_df["system"].map(_pretty_system_name))
    .rename(columns={
        "system": "System",
        "n": "n",
        "routing_accuracy": "Route accuracy",
        "retrieval_precision_at_5": "P@5",
        "raw_hallucination_rate": "Raw hallucination",
        "hallucination_rate": "Unblocked hallucination",
        "task_completion_rate": "Task completion",
        "avg_latency_s": "Avg latency (s)",
    })
    .style.format({
        "Route accuracy": "{:.1%}",
        "P@5": "{:.1%}",
        "Raw hallucination": "{:.1%}",
        "Unblocked hallucination": "{:.1%}",
        "Task completion": "{:.1%}",
        "Avg latency (s)": "{:.2f}",
    }, na_rep="n/a")
)

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    pass

plot_df = summary_df.set_index("system")
metric_cols = [
    ("routing_accuracy", "Route accuracy"),
    ("retrieval_precision_at_5", "Retrieval P@5"),
    ("raw_hallucination_rate", "Raw hallucination"),
    ("hallucination_rate", "Unblocked hallucination"),
    ("task_completion_rate", "Task completion"),
]

fig, axes = plt.subplots(1, 2, figsize=(15, 5), gridspec_kw={"width_ratios": [2.2, 1]})

# Metric comparison chart: higher is better except hallucination rates.
ax = axes[0]
x = np.arange(len(metric_cols))
width = 0.36
systems = list(plot_df.index)
for i, system in enumerate(systems):
    values = [plot_df.loc[system, col] for col, _ in metric_cols]
    plot_values = [0 if pd.isna(v) else v for v in values]
    offset = (i - (len(systems) - 1) / 2) * width
    bars = ax.bar(x + offset, plot_values, width, label=_pretty_system_name(system))
    labels = ["n/a" if pd.isna(v) else f"{v:.0%}" for v in values]
    ax.bar_label(bars, labels=labels, fontsize=8, padding=2)

ax.set_title("Formal Evaluation Metrics")
ax.set_ylabel("Rate")
ax.set_xticks(x)
ax.set_xticklabels([label for _, label in metric_cols], rotation=25, ha="right")
ax.set_ylim(0, 1.12)
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.legend(loc="upper right")

# Latency chart: lower is better.
ax = axes[1]
latency_labels = [_pretty_system_name(s) for s in systems]
latency_values = [plot_df.loc[s, "avg_latency_s"] for s in systems]
bars = ax.bar(latency_labels, latency_values, color=["#4c78a8", "#f58518"][:len(systems)])
ax.bar_label(bars, labels=[f"{v:.2f}s" for v in latency_values], padding=3)
ax.set_title("Average Latency")
ax.set_ylabel("Seconds/query")
ax.tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()

# Optional domain-level charts appear when the full evaluation produced row-level results.
rows = globals().get("formal_eval_rows")
if isinstance(rows, list) and rows:
    rows_df = pd.DataFrame(rows)
    domain_df = (
        rows_df.groupby(["expected_domain", "system"], as_index=False)
        .agg(
            retrieval_precision_at_5=("retrieval_precision_at_5", "mean"),
            task_completion_rate=("task_complete", "mean"),
            avg_latency_s=("latency_s", "mean"),
        )
    )
    display(domain_df.assign(system=domain_df["system"].map(_pretty_system_name)))

    fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
    domains = list(domain_df["expected_domain"].drop_duplicates())
    x = np.arange(len(domains))
    width = 0.36
    for i, system in enumerate(systems):
        subset = domain_df[domain_df["system"] == system].set_index("expected_domain")
        p5 = [subset.loc[d, "retrieval_precision_at_5"] if d in subset.index else np.nan for d in domains]
        latency = [subset.loc[d, "avg_latency_s"] if d in subset.index else np.nan for d in domains]
        offset = (i - (len(systems) - 1) / 2) * width
        axes[0].bar(x + offset, p5, width, label=_pretty_system_name(system))
        axes[1].bar(x + offset, latency, width, label=_pretty_system_name(system))

    axes[0].set_title("Retrieval P@5 by Domain")
    axes[0].set_ylabel("P@5")
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(domains)
    axes[0].set_ylim(0, 1.05)
    axes[0].yaxis.set_major_formatter(PercentFormatter(1.0))
    axes[0].legend()

    axes[1].set_title("Average Latency by Domain")
    axes[1].set_ylabel("Seconds/query")
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(domains)
    axes[1].legend()

    plt.tight_layout()
    plt.show()
else:
    print("Domain-level visualizations will appear after a completed formal evaluation run.")

<a id="22-evaluation-findings"></a>

## 22. Evaluation Findings

- CRAG achieved **100.0%** route accuracy and **100.0%** retrieval P@5.
- CRAG had a higher raw hallucination flag rate (**45.3%**), but validation reduced unblocked hallucination to **4.7%**.
- Monolithic RAG had **19.3%** unblocked hallucination because it has no validation block.
- Task completion was tied at **96.7%** for both systems.
- CRAG was slower: **27.08s/query** vs **5.95s/query**.

Main conclusion: CRAG is useful when safer, validated answers matter more than latency.

Caveat: the labels come from an LLM judge and depend on the current snapshot, prompts, embeddings, and judge settings.

<a id="23-index-validation"></a>

## 23. Operational Health Check

Checks that Chroma collections exist, metadata is populated, and similarity probes return reasonable hits.


In [38]:
# Purpose: reopen the vector store and print final per-domain document counts.
from foundation.vector_store import VectorStoreService
vs = VectorStoreService(embedding_service=embedder)
print(vs.get_all_stats())


[{'domain': 'industrial', 'collection_name': 'industrial_knowledge', 'document_count': 6150}, {'domain': 'recipe', 'collection_name': 'recipe_knowledge', 'document_count': 231311}, {'domain': 'scientific', 'collection_name': 'scientific_knowledge', 'document_count': 686968}]


In [39]:
# Purpose: validate Chroma integrity, metadata coverage, and semantic retrieval quality.
import os, sqlite3
from collections import Counter
from config.settings import CONFIG
EXPECTED = {
    "industrial": {
        "min_docs":      1,
        "meta_required": ["source"],
        "probe":         "PowerFlex 525 fault F004 undervoltage",
        "min_sim":       0.35,
    },
    "recipe": {
        "min_docs":      50,
        "meta_required": ["name", "minutes_bucket", "is_vegetarian", "is_quick"],
        "probe":         "quick weeknight pasta with garlic",
        "min_sim":       0.40,
    },
    "scientific": {
        "min_docs":      1000,
        "meta_required": ["title", "primary_category", "year", "is_ml"],
        "probe":         "retrieval-augmented generation for question answering",
        "min_sim":       0.40,
    },
}
persist_dir = os.path.abspath(
    os.environ.get("CHROMA_PERSIST_DIR") or CONFIG.vector_store.persist_directory
)
db = os.path.join(persist_dir, "chroma.sqlite3")
print("=" * 72)
print(f"Chroma persist dir: {persist_dir}")
if os.path.exists(db):
    print(f"SQLite size:        {os.path.getsize(db) / 1e6:.1f} MB")
    try:
        con = sqlite3.connect(db)
        ok = con.execute("PRAGMA integrity_check;").fetchone()[0]
        con.close()
        print(f"integrity_check:    {ok}")
        if ok != "ok":
            print("WARNING: DB corruption detected. Recover before trusting these results.")
    except Exception as e:
        print(f"integrity_check:    FAILED ({e})")
else:
    print("SQLite: MISSING")
print("=" * 72)
overall_ok = True
for domain, spec in EXPECTED.items():
    print(f"\n--- {domain.upper()} ---")
    n = vs.get_collection_stats(domain)["document_count"]
    print(f"  documents: {n:,}")
    if n < spec["min_docs"]:
        print(f"  FAIL: expected at least {spec['min_docs']:,} docs")
        overall_ok = False
        continue
    coll   = vs._collections[domain]
    sample = coll.get(limit=min(n, 2000), include=["metadatas", "documents"])
    metas  = sample.get("metadatas") or []
    docs   = sample.get("documents") or []
    empty_text = sum(1 for d in docs if not (d or "").strip())
    print(f"  empty text chunks in sample: {empty_text} / {len(docs)}")
    if empty_text > 0:
        print("  WARNING: empty documents present (tokenizer likely substituted zeros)")
    for key in spec["meta_required"]:
        populated = sum(1 for m in metas if m.get(key) not in (None, "", []))
        pct = 100 * populated / max(len(metas), 1)
        status = "OK" if pct >= 80 else "LOW"
        print(f"  meta.{key:<20} populated: {populated}/{len(metas)} ({pct:.0f}%)  [{status}]")
        if pct < 80:
            overall_ok = False
    hits = vs.search(domain, spec["probe"], k=3)
    if not hits:
        print(f"  FAIL: semantic probe returned 0 hits for {spec['probe']!r}")
        overall_ok = False
        continue
    top_sim = max((h.metadata or {}).get("similarity_score", 0) for h in hits)
    print(f"  probe: {spec['probe']!r}  top_sim={top_sim:.3f}")
    if top_sim < spec["min_sim"]:
        print(f"  WARN: top similarity below threshold {spec['min_sim']:.2f}")
        overall_ok = False
    for h in hits[:2]:
        m     = h.metadata or {}
        label = m.get("title") or m.get("name") or m.get("source") or "(no label)"
        print(f"    - sim={m.get('similarity_score', 0):.3f}  {str(label)[:70]}")
print("\n" + "=" * 72)
print("OVERALL:", "PASS" if overall_ok else "ISSUES FOUND")
print("=" * 72)


Chroma persist dir: /content/chroma_db
SQLite size:        13606.5 MB
integrity_check:    ok

--- INDUSTRIAL ---
  documents: 6,150
  empty text chunks in sample: 0 / 2000
  meta.source               populated: 2000/2000 (100%)  [OK]
  probe: 'PowerFlex 525 fault F004 undervoltage'  top_sim=0.934
    - sim=0.934  rockwell_automation
    - sim=0.902  rockwell_automation

--- RECIPE ---
  documents: 231,311
  empty text chunks in sample: 0 / 2000
  meta.name                 populated: 1999/2000 (100%)  [OK]
  meta.minutes_bucket       populated: 2000/2000 (100%)  [OK]
  meta.is_vegetarian        populated: 2000/2000 (100%)  [OK]
  meta.is_quick             populated: 2000/2000 (100%)  [OK]
  probe: 'quick weeknight pasta with garlic'  top_sim=0.938
    - sim=0.938  parmesan garlic pasta
    - sim=0.923  quick pasta for one

--- SCIENTIFIC ---
  documents: 686,968
  empty text chunks in sample: 0 / 2000
  meta.title                populated: 2000/2000 (100%)  [OK]
  meta.primary_category 

<a id="24-final-snapshot"></a>

## 24. Final Snapshot

Optional: save `/content/chroma_db` back to Drive if new indexing was created in this session.


In [40]:
# snapshot_chroma("final")


<a id="25-submission-summary-and-next-steps"></a>

## 25. Submission Summary

This notebook contains the full submission path: setup, indexing controls, workflow build, five-query demo, Basic RAG baseline, learned-component summary, 150-query benchmark, and validation checks.

Reported benchmark result: CRAG is slower but safer. It reduces unblocked hallucination from **19.3%** to **4.7%**, with **96.7%** task completion for both systems.

Submission defaults keep expensive training, indexing, and full evaluation reruns disabled. Re-run those cells only if the corpus, prompts, embeddings, or evaluation settings change.